# Colab GPU Connection Test

This notebook tests the Colab GPU runtime connection from VS Code.

**Instructions:**
1. Open this notebook in VS Code
2. Click **Connect to Google Colab** in the kernel picker (top-right)
3. Select a GPU runtime (A100 or H100 if available on your Colab plan)
4. Run each cell below to verify GPU access

In [42]:
# Cell 1: Basic GPU detection
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Check your Colab runtime type.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU device: NVIDIA A100-SXM4-80GB
GPU memory: 85.1 GB


In [43]:
# Cell 2: OpenMM platform check
try:
    import openmm
    print(f"OpenMM version: {openmm.__version__}")
    print(f"Available platforms:")
    for i in range(openmm.Platform.getNumPlatforms()):
        p = openmm.Platform.getPlatform(i)
        print(f"  {p.getName()}")
    # Try to get CUDA platform
    try:
        cuda = openmm.Platform.getPlatformByName('CUDA')
        print(f"\nCUDA platform available!")
    except Exception:
        print(f"\nCUDA platform NOT available")
except ImportError:
    print("OpenMM not installed in this environment")
    print("Run: !pip install openmm")

OpenMM version: 8.4
Available platforms:
  Reference
  CPU
  OpenCL
  CPU
  CUDA
  OpenCL

CUDA platform available!


In [44]:
# Cell 3: Quick GPU compute test
if torch.cuda.is_available():
    x = torch.randn(1000, 1000, device='cuda')
    y = torch.matmul(x, x)
    print(f"GPU matrix multiply result shape: {y.shape}")
    print(f"GPU compute test: PASSED")
else:
    print("Skipping GPU compute test (no GPU available)")

GPU matrix multiply result shape: torch.Size([1000, 1000])
GPU compute test: PASSED


In [45]:
# Step H - Step 1a: Confirm GPU status
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [46]:
# Step H - Step 1b: Install GPU test dependencies
!pip install openmm-ml torchani 2>&1 | tail -5

ERROR: Could not find a version that satisfies the requirement openmm-ml (from versions: none)
ERROR: No matching distribution found for openmm-ml


In [48]:
# Step H - Step 1c: Check all GPU test dependencies
import openmm
from openmm.app import ForceField

# 1. AMOEBA availability
try:
    ForceField("amoeba2018.xml", "amoeba2018_gk.xml")
    print("AMOEBA: AVAILABLE")
except Exception as e:
    print(f"AMOEBA: NOT AVAILABLE ({e})")

# 2. OpenMM-ML availability
try:
    import openmmml
    print("OpenMM-ML: AVAILABLE")
except ImportError:
    print("OpenMM-ML: NOT AVAILABLE")

# 3. TorchANI availability
try:
    import torchani
    print(f"TorchANI: AVAILABLE (v{torchani.__version__})")
except ImportError:
    print("TorchANI: NOT AVAILABLE")

# 4. CUDA platform
try:
    cuda_plat = openmm.Platform.getPlatformByName('CUDA')
    print("CUDA platform: AVAILABLE")
except Exception:
    print("CUDA platform: NOT AVAILABLE")

# 5. GPU device
import torch
print(f"PyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

AMOEBA: AVAILABLE
OpenMM-ML: AVAILABLE
TorchANI: AVAILABLE (v2.7.9)
CUDA platform: AVAILABLE
PyTorch CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [51]:
# Step H - Step 2a: Create directory structure + write small files
import os
from pathlib import Path

os.makedirs("src/physics", exist_ok=True)
os.makedirs("tests", exist_ok=True)

# src/__init__.py
Path("src/__init__.py").write_text('''"""Core package for the SPINK7-KLK5 molecular dynamics pipeline."""


class PipelineError(Exception):
    """Base exception for all pipeline errors."""


class PhysicalValidityError(PipelineError):
    """Raised when a physical invariant is violated."""


class ConvergenceError(PipelineError):
    """Raised when an iterative solver fails to converge."""


class InsufficientSamplingError(PipelineError):
    """Raised when sampling is insufficient for reliable estimates."""
''')
print("  src/__init__.py")

# src/physics/__init__.py
Path("src/physics/__init__.py").write_text('"""Physics utilities for the SPINK7-KLK5 MD pipeline."""\n')
print("  src/physics/__init__.py")

# tests/__init__.py
Path("tests/__init__.py").write_text('"""Test suite for the SPINK7-KLK5 MD pipeline."""\n')
print("  tests/__init__.py")

# tests/conftest.py (minimal — GPU tests don't need solvation fixtures)
Path("tests/conftest.py").write_text('''"""Shared pytest fixtures for the SPINK7-KLK5 MD pipeline test suite."""

from __future__ import annotations

import pytest


def pytest_configure(config):
    """Register custom pytest markers."""
    config.addinivalue_line(
        "markers",
        "optimized: marks tests that verify behavior under python -O (assert stripped)",
    )
''')
print("  tests/conftest.py (minimal)")
print("All small files written.")

  src/__init__.py
  src/physics/__init__.py
  tests/__init__.py
  tests/conftest.py (minimal)
All small files written.


In [52]:
%%writefile src/config.py
"""Central configuration for the SPINK7-KLK5 MD pipeline.

All physical constants and simulation parameters are defined here.
Units follow the OpenMM internal convention (nm, ps, kJ/mol, K, e).
"""

import os
from dataclasses import dataclass, fields as dataclass_fields
from pathlib import Path

import yaml


# Physical constants
BOLTZMANN_KJ: float = 0.008314462618
AVOGADRO: float = 6.02214076e23
KCAL_TO_KJ: float = 4.184


# Root paths
PROJECT_ROOT: Path = Path(__file__).resolve().parent.parent
DATA_DIR: Path = PROJECT_ROOT / "data"
TRAJ_DIR: Path = DATA_DIR / "trajectories"


SUPPORTED_FORCE_FIELD_FAMILIES: tuple[str, ...] = ("amber", "amoeba", "ml_potential", "qmmm")

SUPPORTED_WATER_MODELS: dict[str, tuple[str, str]] = {
    "tip3p": ("amber14/tip3p.xml", "tip3p"),
    "opc": ("amber14/opc.xml", "tip4pew"),
    "tip4pew": ("amber14/tip4pew.xml", "tip4pew"),
}

SUPPORTED_BOX_SHAPES: tuple[str, ...] = ("cubic", "dodecahedron")


@dataclass(frozen=True)
class SystemConfig:
    """Immutable system preparation parameters."""

    force_field: str = "amber14-all.xml"
    water_model: str = "amber14/tip3p.xml"
    force_field_family: str = "amber"
    ph: float = 7.4
    box_padding_nm: float = 1.2
    ionic_strength_molar: float = 0.15
    positive_ion: str = "Na+"
    negative_ion: str = "Cl-"
    box_shape: str = "cubic"


@dataclass(frozen=True)
class MinimizationConfig:
    """Energy minimization parameters."""

    max_iterations: int = 10_000
    tolerance_kj_mol_nm: float = 10.0


@dataclass(frozen=True)
class EquilibrationConfig:
    """NVT/NPT equilibration parameters."""

    nvt_duration_ps: float = 500.0
    npt_duration_ps: float = 1000.0
    temperature_k: float = 310.0
    friction_per_ps: float = 1.0
    timestep_ps: float = 0.002
    pressure_atm: float = 1.0
    barostat_interval: int = 25
    restraint_k_kj_mol_nm2: float = 1000.0
    save_interval_ps: float = 10.0
    random_seed: int | None = None


@dataclass(frozen=True)
class ProductionConfig:
    """Production MD parameters."""

    duration_ns: float = 100.0
    temperature_k: float = 310.0
    friction_per_ps: float = 1.0
    timestep_ps: float = 0.002
    pressure_atm: float = 1.0
    save_interval_ps: float = 10.0
    checkpoint_interval_ps: float = 100.0
    random_seed: int | None = None


@dataclass(frozen=True)
class SMDConfig:
    """Steered Molecular Dynamics parameters."""

    spring_constant_kj_mol_nm2: float = 1000.0
    pulling_velocity_nm_per_ps: float = 0.001
    pull_distance_nm: float = 3.0
    n_replicates: int = 50
    save_interval_ps: float = 1.0
    random_seed: int | None = None


@dataclass(frozen=True)
class UmbrellaConfig:
    """Umbrella Sampling parameters."""

    xi_min_nm: float = 1.5
    xi_max_nm: float = 4.0
    window_spacing_nm: float = 0.05
    spring_constant_kj_mol_nm2: float = 1000.0
    per_window_duration_ns: float = 10.0
    save_interval_ps: float = 1.0
    pre_position_velocity_nm_per_ps: float = 0.01
    pre_position_spring_constant_kj_mol_nm2: float = 1000.0
    equilibration_duration_ps: float = 200.0
    detect_equilibration: bool = True


@dataclass(frozen=True)
class WHAMConfig:
    """WHAM solver parameters."""

    tolerance: float = 1e-6
    max_iterations: int = 100_000
    n_bootstrap: int = 200
    histogram_bins: int = 200


@dataclass(frozen=True)
class MetadynamicsConfig:
    """Well-tempered metadynamics parameters."""

    gaussian_height_kj_mol: float = 1.0
    gaussian_width_nm: float = 0.05
    deposition_interval_ps: float = 1.0
    bias_factor: float = 15.0
    temperature_k: float = 310.0
    simulation_duration_ns: float = 50.0
    save_interval_ps: float = 1.0
    convergence_tolerance_kj_mol: float = 1.0
    convergence_check_interval_ns: float = 1.0
    grid_min_nm: float = 1.0
    grid_max_nm: float = 5.0
    grid_num_bins: int = 200


@dataclass(frozen=True)
class MBARConfig:
    """MBAR solver parameters."""

    solver_protocol: str = "robust"
    relative_tolerance: float = 1e-7
    maximum_iterations: int = 10_000
    n_bootstrap: int = 200
    n_pmf_bins: int = 200


@dataclass(frozen=True)
class FEPConfig:
    """Alchemical free energy perturbation parameters."""

    n_lambda_windows: int = 20
    per_window_duration_ns: float = 2.0
    temperature_k: float = 310.0
    soft_core_alpha: float = 0.5
    soft_core_power: int = 1
    save_interval_ps: float = 1.0
    n_equilibration_steps: int = 5000
    annihilate_electrostatics: bool = True
    annihilate_sterics: bool = False


@dataclass(frozen=True)
class AMOEBAConfig:
    """AMOEBA polarizable force field parameters."""

    force_field_xml: str = "amoeba2018.xml"
    water_model_xml: str = "amoeba2018_gk.xml"
    mutual_induced_target_epsilon: float = 1e-5
    polarization_type: str = "mutual"
    pme_grid_spacing_nm: float = 0.06
    ewald_error_tolerance: float = 1e-5


@dataclass(frozen=True)
class MLPotentialConfig:
    """Machine-learned force field parameters."""

    potential_name: str = "ani2x"
    implementation: str = "torchani"
    mixed_precision: bool = True


@dataclass(frozen=True)
class QMMMConfig:
    """QM/MM hybrid potential parameters (stub)."""

    qm_method: str = "B3LYP"
    qm_basis_set: str = "6-31G*"
    qm_region_residues: tuple[str, ...] = ("HIS57", "ASP102", "SER195")


@dataclass(frozen=True)
class FiniteSizeCorrectionConfig:
    """Finite-size electrostatic correction parameters."""

    ewald_self_energy_constant: float = -2.837297
    solvent_dielectric: float = 80.0
    coulomb_constant_kj_nm_per_mol_e2: float = 138.935456
    apply_correction: bool = True


@dataclass(frozen=True)
class MSMConfig:
    """Markov State Model construction parameters."""

    tica_lag_ps: float = 10.0
    tica_n_components: int = 4
    n_clusters: int = 200
    lag_times_ps: tuple[float, ...] = (1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 500.0)
    n_implied_timescales: int = 5
    bayesian_n_samples: int = 100
    stride: int = 1


# ---------------------------------------------------------------------------
# Configuration loading (L-21)
# ---------------------------------------------------------------------------

_SECTION_TO_DATACLASS: dict[str, type] = {
    "system": SystemConfig,
    "minimization": MinimizationConfig,
    "equilibration": EquilibrationConfig,
    "production": ProductionConfig,
    "smd": SMDConfig,
    "umbrella": UmbrellaConfig,
    "wham": WHAMConfig,
    "mbar": MBARConfig,
    "amoeba": AMOEBAConfig,
    "ml_potential": MLPotentialConfig,
    "qmmm": QMMMConfig,
    "finite_size_correction": FiniteSizeCorrectionConfig,
    "fep": FEPConfig,
    "metadynamics": MetadynamicsConfig,
    "msm": MSMConfig,
}


def _build_dataclass_from_dict(dataclass_type: type, overrides: dict) -> object:
    """Construct a frozen dataclass, applying overrides on top of defaults."""
    valid_field_names = {f.name for f in dataclass_fields(dataclass_type)}
    unknown_keys = set(overrides.keys()) - valid_field_names
    if unknown_keys:
        raise TypeError(
            f"Unrecognized fields for {dataclass_type.__name__}: {sorted(unknown_keys)}"
        )

    kwargs = {}
    for field in dataclass_fields(dataclass_type):
        if field.name not in overrides:
            continue
        value = overrides[field.name]
        if value is None:
            kwargs[field.name] = None
            continue
        default_val = field.default
        if default_val is not dataclass_fields.__class__:
            if isinstance(default_val, float) and isinstance(value, (int, float)):
                value = float(value)
            elif isinstance(default_val, tuple) and isinstance(value, list):
                value = tuple(value)
        kwargs[field.name] = value

    return dataclass_type(**kwargs)


_ENV_PREFIX = "SPINK7"


def _apply_env_overrides(section_name: str, config_instance: object) -> object:
    """Override dataclass fields from environment variables."""
    from dataclasses import replace

    overrides = {}
    for field in dataclass_fields(config_instance):
        env_key = f"{_ENV_PREFIX}_{section_name}_{field.name}".upper()
        env_value = os.environ.get(env_key)
        if env_value is not None:
            default_val = field.default
            if isinstance(default_val, float):
                overrides[field.name] = float(env_value)
            elif isinstance(default_val, int):
                overrides[field.name] = int(env_value)
            elif isinstance(default_val, bool):
                overrides[field.name] = env_value.lower() in ("true", "1", "yes")
            elif default_val is None:
                try:
                    overrides[field.name] = int(env_value)
                except ValueError:
                    try:
                        overrides[field.name] = float(env_value)
                    except ValueError:
                        overrides[field.name] = env_value
            else:
                overrides[field.name] = env_value

    if overrides:
        return replace(config_instance, **overrides)
    return config_instance


def load_config(config_path: str | Path | None = None) -> dict[str, object]:
    """Load simulation configuration from a YAML file."""
    if config_path is None:
        configs = {name: cls() for name, cls in _SECTION_TO_DATACLASS.items()}
        for section_name in configs:
            configs[section_name] = _apply_env_overrides(section_name, configs[section_name])
        return configs

    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Configuration file not found: {path}")

    with path.open("r", encoding="utf-8") as handle:
        raw = yaml.safe_load(handle)

    if raw is None:
        raw = {}

    configs: dict[str, object] = {}
    for section_name, dataclass_type in _SECTION_TO_DATACLASS.items():
        section_data = raw.get(section_name, {})
        if section_data is None:
            section_data = {}
        configs[section_name] = _build_dataclass_from_dict(dataclass_type, section_data)

    for section_name in configs:
        configs[section_name] = _apply_env_overrides(section_name, configs[section_name])

    return configs

Writing src/config.py


In [53]:
%%writefile src/physics/force_field_factory.py
"""Force field factory for the SPINK7-KLK5 MD pipeline.

Provides a unified interface for creating OpenMM System objects from
different force field families (AMBER, AMOEBA, ML potentials, QM/MM).
"""

from __future__ import annotations

import logging

import openmm
import openmm.app
from openmm import unit
from openmm.app import ForceField, HBonds, PME, Topology

from src.config import (
    SUPPORTED_FORCE_FIELD_FAMILIES,
    AMOEBAConfig,
    MLPotentialConfig,
    QMMMConfig,
    SystemConfig,
)

logger = logging.getLogger(__name__)


def create_system(
    topology: Topology,
    positions: unit.Quantity,
    system_config: SystemConfig,
    amoeba_config: AMOEBAConfig | None = None,
    ml_config: MLPotentialConfig | None = None,
    qmmm_config: QMMMConfig | None = None,
) -> openmm.System:
    """Create an OpenMM System from the specified force field family.

    Args:
        topology: OpenMM Topology object.
        positions: Atomic positions with units.
        system_config: System preparation parameters (includes force_field_family).
        amoeba_config: Required if force_field_family == 'amoeba'.
        ml_config: Required if force_field_family == 'ml_potential'.
        qmmm_config: Required if force_field_family == 'qmmm'.

    Returns:
        Parameterized OpenMM System object.

    Raises:
        ValueError: If force_field_family is not in SUPPORTED_FORCE_FIELD_FAMILIES.
        ValueError: If required config is not provided for the selected family.
    """
    family = system_config.force_field_family

    if family not in SUPPORTED_FORCE_FIELD_FAMILIES:
        raise ValueError(
            f"Unsupported force_field_family '{family}'. "
            f"Must be one of {SUPPORTED_FORCE_FIELD_FAMILIES}"
        )

    if family == "amber":
        return _create_amber_system(topology, system_config)
    elif family == "amoeba":
        if amoeba_config is None:
            raise ValueError(
                "amoeba_config is required when force_field_family == 'amoeba'"
            )
        return _create_amoeba_system(topology, amoeba_config)
    elif family == "ml_potential":
        if ml_config is None:
            raise ValueError(
                "ml_config is required when force_field_family == 'ml_potential'"
            )
        return _create_ml_system(topology, positions, ml_config)
    elif family == "qmmm":
        if qmmm_config is None:
            qmmm_config = QMMMConfig()
        return _create_qmmm_system(topology, positions, qmmm_config)

    raise ValueError(f"Unhandled force_field_family: {family}")  # pragma: no cover


def _create_amber_system(
    topology: Topology,
    system_config: SystemConfig,
) -> openmm.System:
    """Create an AMBER ff14SB system (existing pipeline default)."""
    force_field = ForceField(system_config.force_field, system_config.water_model)
    return force_field.createSystem(
        topology,
        nonbondedMethod=PME,
        nonbondedCutoff=1.0 * unit.nanometers,
        constraints=HBonds,
        rigidWater=True,
    )


def _create_amoeba_system(
    topology: Topology,
    amoeba_config: AMOEBAConfig,
) -> openmm.System:
    """Create an AMOEBA polarizable system.

    Detects implicit solvent models (Generalized Kirkwood) and adjusts the
    nonbonded method accordingly: implicit solvent requires NoCutoff, while
    explicit solvent uses PME.
    """
    from openmm.app import NoCutoff

    force_field = ForceField(
        amoeba_config.force_field_xml,
        amoeba_config.water_model_xml,
    )

    is_implicit = "_gk" in amoeba_config.water_model_xml.lower()

    create_kwargs: dict[str, object] = {
        "polarization": amoeba_config.polarization_type,
        "mutualInducedTargetEpsilon": amoeba_config.mutual_induced_target_epsilon,
    }

    if is_implicit:
        create_kwargs["nonbondedMethod"] = NoCutoff
    else:
        create_kwargs["nonbondedMethod"] = PME
        create_kwargs["nonbondedCutoff"] = 0.7 * unit.nanometers

    return force_field.createSystem(topology, **create_kwargs)


def validate_amoeba_dipole_convergence(
    simulation: openmm.app.Simulation,
    max_dipole_norm: float = 10.0,
) -> bool:
    """Verify that AMOEBA induced dipoles converged to physical values."""
    import numpy as np

    try:
        state = simulation.context.getState(getEnergy=True)
        energy = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        if not np.isfinite(energy):
            logger.warning("AMOEBA energy is not finite — dipole convergence failure likely")
            return False
        return True
    except Exception:
        logger.warning("Failed to evaluate AMOEBA energy for dipole convergence check")
        return False


def _create_ml_system(
    topology: Topology,
    positions: unit.Quantity,
    ml_config: MLPotentialConfig,
) -> openmm.System:
    """Create a machine-learned potential system via openmmml."""
    try:
        from openmmml import MLPotential
    except ImportError as exc:
        raise ImportError(
            "openmmml package is required for ML force fields. "
            "Install with: pip install openmm-ml"
        ) from exc

    potential = MLPotential(ml_config.potential_name)
    return potential.createSystem(topology, implementation=ml_config.implementation)


def _create_qmmm_system(
    topology: Topology,
    positions: unit.Quantity,
    qmmm_config: QMMMConfig,
) -> openmm.System:
    """Create a QM/MM system (not yet implemented)."""
    raise NotImplementedError(
        "QM/MM force field integration is not yet implemented. "
        "This placeholder defines the interface for future development. "
        "See L-15 implementation guide, Step 5 for design specifications."
    )


def compare_force_field_energies(
    topology: Topology,
    positions: unit.Quantity,
    system_config_amber: SystemConfig,
    system_config_alt: SystemConfig,
    amoeba_config: AMOEBAConfig | None = None,
    ml_config: MLPotentialConfig | None = None,
) -> dict[str, object]:
    """Compute potential energies with both AMBER and an alternative force field."""
    import numpy as np

    amber_system = create_system(topology, positions, system_config_amber)
    amber_integrator = openmm.VerletIntegrator(0.001 * unit.picoseconds)
    platform = openmm.Platform.getPlatformByName("CPU")
    amber_sim = openmm.app.Simulation(topology, amber_system, amber_integrator, platform)
    amber_sim.context.setPositions(positions)
    amber_energy = (
        amber_sim.context.getState(getEnergy=True)
        .getPotentialEnergy()
        .value_in_unit(unit.kilojoule_per_mole)
    )

    alt_system = create_system(
        topology, positions, system_config_alt,
        amoeba_config=amoeba_config, ml_config=ml_config,
    )
    alt_integrator = openmm.VerletIntegrator(0.001 * unit.picoseconds)
    alt_sim = openmm.app.Simulation(topology, alt_system, alt_integrator, platform)
    alt_sim.context.setPositions(positions)
    alt_energy = (
        alt_sim.context.getState(getEnergy=True)
        .getPotentialEnergy()
        .value_in_unit(unit.kilojoule_per_mole)
    )

    return {
        "amber_energy_kj_mol": float(amber_energy),
        "alt_energy_kj_mol": float(alt_energy),
        "difference_kj_mol": float(alt_energy - amber_energy),
        "alt_family": system_config_alt.force_field_family,
        "both_finite": bool(np.isfinite(amber_energy) and np.isfinite(alt_energy)),
    }

Writing src/physics/force_field_factory.py


In [54]:
%%writefile tests/test_force_field_factory.py
"""Tests for the force field factory module."""

from __future__ import annotations

from unittest.mock import MagicMock

import numpy as np
import openmm
import pytest
from openmm import unit
from openmm.app import ForceField, HBonds, PME, Simulation

from src.config import (
    AMOEBAConfig,
    MLPotentialConfig,
    QMMMConfig,
    SystemConfig,
)
from src.physics.force_field_factory import create_system


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def _amoeba_available() -> bool:
    """Check whether AMOEBA XML files are accessible via OpenMM."""
    try:
        ForceField("amoeba2018.xml", "amoeba2018_gk.xml")
        return True
    except Exception:
        return False


def _openmmml_available() -> bool:
    """Check whether openmmml/torchani is installed."""
    try:
        import openmmml  # noqa: F401
        return True
    except ImportError:
        return False


def _build_unsolvated_alanine_dipeptide():
    """Build a minimal unsolvated alanine dipeptide topology for AMOEBA testing.

    Returns (topology, positions) without any explicit water, suitable for
    implicit-solvent (GK) AMOEBA calculations. Uses AMBER force field for
    hydrogen placement since AMOEBA's addHydrogens has OpenMM limitations.
    """
    from openmm.app import PDBFile, Modeller
    import tempfile, os

    pdb_content = """\
ATOM      1  CH3 ACE A   1      -2.300   1.200   0.000  1.00 20.00           C
ATOM      2  C   ACE A   1      -1.000   0.800   0.000  1.00 20.00           C
ATOM      3  O   ACE A   1      -0.300   1.700   0.000  1.00 20.00           O
ATOM      4  N   ALA A   2      -0.650  -0.500   0.000  1.00 20.00           N
ATOM      5  CA  ALA A   2       0.700  -1.050   0.000  1.00 20.00           C
ATOM      6  C   ALA A   2       1.850  -0.050   0.000  1.00 20.00           C
ATOM      7  O   ALA A   2       2.980  -0.430   0.000  1.00 20.00           O
ATOM      8  CB  ALA A   2       0.820  -2.570   0.000  1.00 20.00           C
ATOM      9  N   NME A   3       1.560   1.210   0.000  1.00 20.00           N
ATOM     10  CH3 NME A   3       2.540   2.220   0.000  1.00 20.00           C
TER
END
"""
    fd, path = tempfile.mkstemp(suffix=".pdb")
    try:
        os.write(fd, pdb_content.encode())
        os.close(fd)
        pdb = PDBFile(path)
        amber_ff = ForceField("amber14-all.xml")
        modeller = Modeller(pdb.topology, pdb.positions)
        modeller.addHydrogens(amber_ff)
        return modeller.topology, modeller.positions
    finally:
        os.unlink(path)


# ---------------------------------------------------------------------------
# L-15 Step 2: Core factory tests
# ---------------------------------------------------------------------------


def test_create_system_amber_produces_valid_system(alanine_dipeptide_simulation):
    """AMBER factory output should produce a system with NonbondedForce (PME)."""
    sim = alanine_dipeptide_simulation
    topology = sim.topology
    positions = sim.context.getState(getPositions=True).getPositions()

    system = create_system(topology, positions, SystemConfig())

    assert system.getNumParticles() > 0
    has_nonbonded = any(
        isinstance(system.getForce(i), openmm.NonbondedForce)
        for i in range(system.getNumForces())
    )
    assert has_nonbonded, "AMBER system must use NonbondedForce with PME"


def test_create_system_rejects_unknown_family():
    """Factory should reject unsupported force field families."""
    mock_topology = MagicMock()
    mock_positions = MagicMock()

    config = SystemConfig(force_field_family="unknown")
    with pytest.raises(ValueError, match="force_field_family"):
        create_system(mock_topology, mock_positions, config)


def test_create_system_amoeba_requires_config():
    """Factory should raise ValueError if amoeba_config is missing for AMOEBA family."""
    mock_topology = MagicMock()
    mock_positions = MagicMock()

    config = SystemConfig(force_field_family="amoeba")
    with pytest.raises(ValueError, match="amoeba_config"):
        create_system(mock_topology, mock_positions, config)


def test_create_system_ml_requires_config():
    """Factory should raise ValueError if ml_config is missing for ML family."""
    mock_topology = MagicMock()
    mock_positions = MagicMock()

    config = SystemConfig(force_field_family="ml_potential")
    with pytest.raises(ValueError, match="ml_config"):
        create_system(mock_topology, mock_positions, config)


# ---------------------------------------------------------------------------
# L-15 Step 3: AMOEBA integration
# ---------------------------------------------------------------------------


@pytest.mark.skipif(
    not _amoeba_available(),
    reason="AMOEBA force field XML files not available",
)
def test_amoeba_system_produces_finite_energy():
    """AMOEBA system should compute a finite potential energy."""
    topology, positions = _build_unsolvated_alanine_dipeptide()

    config = SystemConfig(force_field_family="amoeba")
    amoeba_config = AMOEBAConfig()

    system = create_system(topology, positions, config, amoeba_config=amoeba_config)

    integrator = openmm.LangevinMiddleIntegrator(
        310 * unit.kelvin, 1.0 / unit.picosecond, 0.001 * unit.picoseconds,
    )
    platform = openmm.Platform.getPlatformByName("CPU")
    test_sim = Simulation(topology, system, integrator, platform)
    test_sim.context.setPositions(positions)
    state = test_sim.context.getState(getEnergy=True)
    energy = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)

    assert np.isfinite(energy), "AMOEBA potential energy must be finite"


# ---------------------------------------------------------------------------
# L-15 Step 4: ML force field integration
# ---------------------------------------------------------------------------


@pytest.mark.skipif(
    not _openmmml_available(),
    reason="openmmml/torchani not installed",
)
def test_ml_potential_creates_valid_system():
    """ANI-2x via openmmml should create a valid OpenMM System with TorchForce.

    Validates that the openmmml -> torchani -> TorchForce pipeline produces
    a well-formed System object. Runs the actual system creation in a
    subprocess to prevent the openmmtorch exit-cleanup segfault (macOS ARM
    pip install) from crashing the pytest process.
    """
    import subprocess
    import sys
    import textwrap

    script = textwrap.dedent("""\
        import sys, os
        sys.path.insert(0, os.getcwd())
        from src.config import SystemConfig, MLPotentialConfig
        from src.physics.force_field_factory import create_system
        from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide

        topology, positions = _build_unsolvated_alanine_dipeptide()
        config = SystemConfig(force_field_family="ml_potential")
        ml_config = MLPotentialConfig(potential_name="ani2x")
        system = create_system(topology, positions, config, ml_config=ml_config)

        n_particles = system.getNumParticles()
        n_forces = system.getNumForces()
        force_types = [type(system.getForce(i)).__name__ for i in range(n_forces)]

        print(f"PARTICLES={n_particles}")
        print(f"FORCES={n_forces}")
        print(f"FORCE_TYPES={','.join(force_types)}")
        print("SUCCESS")
    """)

    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True,
        text=True,
        timeout=300,
        cwd=str(__import__("pathlib").Path(__file__).resolve().parent.parent),
    )

    stdout = result.stdout
    assert "SUCCESS" in stdout, (
        f"ML system creation failed.\nstdout: {stdout}\nstderr: {result.stderr[-500:]}"
    )
    assert "PARTICLES=22" in stdout, "System should have 22 particles"

    for line in stdout.splitlines():
        if line.startswith("FORCE_TYPES="):
            force_types = line.split("=", 1)[1].split(",")
            assert any(ft in ("Force", "TorchForce") for ft in force_types), (
                f"Expected TorchForce in system, got: {force_types}"
            )


@pytest.mark.skipif(
    not _openmmml_available(),
    reason="openmmml/torchani not installed",
)
def test_ml_potential_ani2x_direct_energy():
    """ANI-2x should compute finite energies for alanine dipeptide via torchani directly.

    Bypasses the OpenMM Context/TorchForce kernel to validate ANI-2x energy
    evaluation independently. This confirms the neural network potential
    produces physically reasonable energies for organic molecules.
    """
    import torch
    import torchani

    topology, positions = _build_unsolvated_alanine_dipeptide()

    elements_to_z = {"hydrogen": 1, "carbon": 6, "nitrogen": 7, "oxygen": 8}
    species = []
    coords = []
    for atom in topology.atoms():
        z = elements_to_z.get(atom.element.name.lower())
        if z is None:
            pytest.skip(f"Unsupported element for ANI-2x: {atom.element.name}")
        species.append(z)

    for i in range(len(species)):
        pos = positions[i]
        coords.append([
            pos[0].value_in_unit(unit.angstrom),
            pos[1].value_in_unit(unit.angstrom),
            pos[2].value_in_unit(unit.angstrom),
        ])

    species_tensor = torch.tensor([species], dtype=torch.long)
    coords_tensor = torch.tensor([coords], dtype=torch.float32, requires_grad=True)

    model = torchani.models.ANI2x(periodic_table_index=True)
    energy = model((species_tensor, coords_tensor)).energies
    energy_val = energy.item()

    assert np.isfinite(energy_val), f"ANI-2x energy must be finite, got {energy_val}"
    assert energy_val < 0, f"ANI-2x energy should be negative for a stable molecule, got {energy_val}"


@pytest.mark.skipif(
    not _openmmml_available(),
    reason="openmmml/torchani not installed",
)
def test_ml_potential_produces_finite_energy():
    """ANI-2x ML potential should compute a finite energy via OpenMM Context.

    Uses an unsolvated alanine dipeptide (H, C, N, O atoms only) and runs
    the full pipeline (system creation + Context + energy evaluation) in a
    subprocess to isolate the openmmtorch exit-cleanup segfault on macOS ARM.
    """
    import subprocess
    import sys
    import textwrap

    script = textwrap.dedent("""\
        import sys, os
        sys.path.insert(0, os.getcwd())
        import openmm
        from openmm import unit
        from openmm.app import Simulation
        from src.config import SystemConfig, MLPotentialConfig
        from src.physics.force_field_factory import create_system
        from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide

        sp = os.path.dirname(os.path.dirname(openmm.__file__))
        plugin_dir = os.path.join(sp, "OpenMM.libs", "lib", "plugins")
        if os.path.isdir(plugin_dir):
            openmm.Platform.loadPluginsFromDirectory(plugin_dir)

        topology, positions = _build_unsolvated_alanine_dipeptide()
        config = SystemConfig(force_field_family="ml_potential")
        ml_config = MLPotentialConfig(potential_name="ani2x")
        system = create_system(topology, positions, config, ml_config=ml_config)

        integrator = openmm.VerletIntegrator(0.001 * unit.picoseconds)
        sim = Simulation(topology, system, integrator)
        sim.context.setPositions(positions)
        state = sim.context.getState(getEnergy=True)
        energy = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)

        import math
        if not math.isfinite(energy):
            print(f"ENERGY_ERROR={energy}")
            sys.exit(1)
        print(f"ENERGY={energy}")
        print("SUCCESS")
    """)

    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True,
        text=True,
        timeout=300,
        cwd=str(__import__("pathlib").Path(__file__).resolve().parent.parent),
    )

    stdout = result.stdout
    stderr = result.stderr

    if "SUCCESS" not in stdout and result.returncode != 0:
        pytest.skip(
            f"OpenMM Context with TorchForce not functional "
            f"(exit={result.returncode}): {stderr[-300:]}"
        )

    assert "SUCCESS" in stdout, (
        f"Energy evaluation failed.\nstdout: {stdout}\nstderr: {stderr[-500:]}"
    )


def test_ml_potential_import_error_message():
    """Factory should raise ImportError with install instructions when openmmml missing."""
    import sys
    import importlib

    if _openmmml_available():
        pytest.skip("openmmml is installed; cannot test ImportError path")

    mock_topology = MagicMock()
    mock_positions = MagicMock()

    config = SystemConfig(force_field_family="ml_potential")
    ml_config = MLPotentialConfig()

    with pytest.raises(ImportError, match="openmm-ml"):
        create_system(mock_topology, mock_positions, config, ml_config=ml_config)


# ---------------------------------------------------------------------------
# L-15 Step 5: QM/MM stub
# ---------------------------------------------------------------------------


def test_qmmm_raises_not_implemented():
    """QM/MM backend should raise NotImplementedError with clear message."""
    mock_topology = MagicMock()
    mock_positions = MagicMock()

    config = SystemConfig(force_field_family="qmmm")
    with pytest.raises(NotImplementedError, match="QM/MM.*not yet implemented"):
        create_system(mock_topology, mock_positions, config, qmmm_config=QMMMConfig())

Writing tests/test_force_field_factory.py


In [55]:
# Step H - Step 2b: Verify files and imports
import os, sys

# Ensure project root is on sys.path
project_root = "/content"
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")

# Check all required files
required_files = [
    "src/physics/force_field_factory.py",
    "src/config.py",
    "src/__init__.py",
    "tests/test_force_field_factory.py",
]
all_found = True
for f in required_files:
    path = os.path.join(project_root, f)
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {f}")
    if not exists:
        all_found = False

# Verify imports
print("\nImport verification:")
try:
    from src.config import SystemConfig, AMOEBAConfig, MLPotentialConfig
    print("  ✅ SystemConfig, AMOEBAConfig, MLPotentialConfig")
except Exception as e:
    print(f"  ❌ Config imports failed: {e}")

try:
    from src.physics.force_field_factory import create_system
    print("  ✅ create_system")
except Exception as e:
    print(f"  ❌ create_system import failed: {e}")

print(f"\nAll files found: {all_found}")
print("All imports successful: True")

  ✅ src/physics/force_field_factory.py
  ✅ src/config.py
  ✅ src/__init__.py
  ✅ tests/test_force_field_factory.py

Import verification:
  ✅ SystemConfig, AMOEBAConfig, MLPotentialConfig
  ✅ create_system

All files found: True
All imports successful: True


In [56]:
# Step H - Step 4: Execute GPU-02 — ANI-2x ML Potential System Creation via OpenMM-ML
# Run test_ml_potential_creates_valid_system via pytest
import subprocess, sys, datetime

print(f"GPU-02 Execution Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "test_ml_potential_creates_valid_system",
     "-v", "--tb=long"],
    capture_output=True, text=True, timeout=600,
    cwd="/content"
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR (last 1000 chars):")
print(result.stderr[-1000:] if result.stderr else "(empty)")
print(f"\nReturn code: {result.returncode}")

# Per implementation guide Step 4 note:
# Exit code 139 (segfault) with "SUCCESS" in subprocess stdout is expected
# openmmtorch cleanup behavior — test should still PASS
if "1 passed" in result.stdout:
    print("\n✅ GPU-02: PASSED")
elif result.returncode == 139 and "PASSED" in result.stdout:
    print("\n✅ GPU-02: PASSED (exit code 139 — expected openmmtorch cleanup segfault)")
elif "1 skipped" in result.stdout:
    print("\n⏭️ GPU-02: SKIPPED — openmmml/torchani not available")
else:
    print("\n❌ GPU-02: FAILED — see output above")

GPU-02 Execution Timestamp: 2026-03-20 22:48
STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.7.18, anyio-4.12.1, typeguard-4.5.1
collecting ... collected 10 items / 9 deselected / 1 selected

tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [100%]

=================================== FAILURES ===================================
____________________ test_ml_potential_creates_valid_system ____________________

    @pytest.mark.skipif(
        not _openmmml_available(),
        reason="openmmml/torchani not installed",
    )
    def test_ml_potential_creates_valid_system():
        """ANI-2x via openmmml should create a valid OpenMM System with TorchForce.
    
        Validates that the openmmml -> torchani -> TorchForce pipeline produces
        a well-formed System object.

In [57]:
# GPU-02 Result Summary (compact output extraction)
print("=" * 50)
print("GPU-02 RESULT SUMMARY")
print("=" * 50)
print(f"Return code: {result.returncode}")
print()

# Extract key lines from stdout
for line in result.stdout.splitlines():
    line_stripped = line.strip()
    if any(kw in line_stripped for kw in [
        "PASSED", "FAILED", "SKIPPED", "ERROR",
        "PARTICLES=", "FORCES=", "FORCE_TYPES=", "SUCCESS",
        "passed", "failed", "skipped", "error", "warnings summary",
        "1 passed", "test_ml_potential"
    ]):
        print(f"  {line_stripped}")

print()
# Show last 5 lines of stderr if any
if result.stderr:
    stderr_lines = result.stderr.strip().splitlines()
    print(f"STDERR (last 5 lines of {len(stderr_lines)} total):")
    for line in stderr_lines[-5:]:
        print(f"  {line}")

# Final verdict
if "1 passed" in result.stdout:
    print("\n✅ GPU-02: PASSED")
elif "PASSED" in result.stdout and result.returncode in (0, 139, -11):
    print("\n✅ GPU-02: PASSED (with segfault cleanup — expected)")
elif "1 skipped" in result.stdout:
    print("\n⏭️ GPU-02: SKIPPED")
else:
    print("\n❌ GPU-02: FAILED")

GPU-02 RESULT SUMMARY
Return code: 1

  tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [100%]
  ____________________ test_ml_potential_creates_valid_system ____________________
  def test_ml_potential_creates_valid_system():
  print(f"PARTICLES={n_particles}")
  print(f"FORCES={n_forces}")
  print(f"FORCE_TYPES={','.join(force_types)}")
  print("SUCCESS")
  >       assert "SUCCESS" in stdout, (
  f"ML system creation failed.\nstdout: {stdout}\nstderr: {result.stderr[-500:]}"
  E       AssertionError: ML system creation failed.
  E             raise ImportError(f"Failed to import torchani with error: {e}. Make sure torchani is installed.")
  E         ImportError: Failed to import torchani with error: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: _ZNK2at7Context14allowTF32CuDNNEv. Make sure torchani is installed.
  E       assert 'SUCCESS' in ''
  FAILED tests/test_force_field_factory.py::test_ml_potential_creates

In [58]:
# GPU-02 Troubleshooting: Diagnose PyTorch/TorchANI version mismatch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version (PyTorch built with): {torch.version.cuda}")

# Check torchani version
try:
    import torchani
    print(f"TorchANI version: {torchani.__version__}")
except ImportError as e:
    print(f"TorchANI import error: {e}")

# Check openmm-torch version
try:
    import openmmtorch
    print(f"OpenMM-Torch: available")
except ImportError as e:
    print(f"OpenMM-Torch import error: {e}")

# Check openmm-ml version
try:
    import openmmml
    print(f"OpenMM-ML version: {openmmml.__version__}")
except ImportError as e:
    print(f"OpenMM-ML import error: {e}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version (PyTorch built with): 12.8
TorchANI version: 2.7.9
OpenMM-Torch import error: /lib/x86_64-linux-gnu/libstdc++.so.6: version `CXXABI_1.3.15' not found (required by /opt/conda/envs/gpu/lib/python3.12/site-packages/_openmmtorch.cpython-312-x86_64-linux-gnu.so)


AttributeError: module 'openmmml' has no attribute '__version__'

In [59]:
# GPU-02 Troubleshooting: Check subprocess environment
import sys, subprocess

print(f"Kernel sys.executable: {sys.executable}")

# Check what the subprocess sees
diag_script = """
import sys, os
print(f"SUB sys.executable: {sys.executable}")
print(f"SUB sys.path[:5]: {sys.path[:5]}")
print(f"SUB torch path: {__import__('torch').__file__}")
print(f"SUB torch version: {__import__('torch').__version__}")
try:
    import torchani
    print(f"SUB torchani: OK version={torchani.__version__}")
except Exception as e:
    print(f"SUB torchani: FAILED {e}")
"""

r = subprocess.run([sys.executable, "-c", diag_script],
                   capture_output=True, text=True, timeout=30)
print(r.stdout)
if r.stderr:
    stderr_lines = r.stderr.strip().splitlines()
    print("STDERR (last 3):", stderr_lines[-3:] if len(stderr_lines) >= 3 else stderr_lines)

Kernel sys.executable: /usr/bin/python3
SUB sys.executable: /usr/bin/python3
SUB sys.path[:5]: ['', '/usr/local/lib/python3.12/dist-packages', '/opt/conda/envs/gpu/lib/python3.12/site-packages', '/usr/lib/python312.zip', '/usr/lib/python3.12']
SUB torch path: /usr/local/lib/python3.12/dist-packages/torch/__init__.py
SUB torch version: 2.8.0+cu128
SUB torchani: OK version=2.7.9

STDERR (last 3): ["/usr/local/lib/python3.12/dist-packages/torchani/csrc/__init__.py:56: UserWarning: The extensions: ['cuaev', 'mnp', 'cell_list'] are not installed and will not be available. To install the extensions first install the CUDA Toolkit, and afterwards  run `ani build-extensions` To suppress warn set the env var TORCHANI_NO_WARN_EXTENSIONS=1 For example, if using bash, you may add `export TORCHANI_NO_WARN_EXTENSIONS=1` to your .bashrc", '  warnings.warn(']


In [60]:
# GPU-02 Fix: Reinstall torchani to compile against runtime PyTorch
# The subprocess uses /usr/local torch (2.8.0), so we rebuild torchani for it
import subprocess, sys

# Force reinstall torchani to build against the current torch 
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "torchani"],
    capture_output=True, text=True, timeout=300
)
# Show last 10 lines of output
lines = r.stdout.strip().splitlines()
for line in lines[-10:]:
    print(line)
print(f"Return code: {r.returncode}")

# Also reinstall openmm-ml to ensure compatibility
r2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir", "openmm-ml"],
    capture_output=True, text=True, timeout=300
)
lines2 = r2.stdout.strip().splitlines()
for line in lines2[-10:]:
    print(line)
print(f"openmm-ml Return code: {r2.returncode}")

      Successfully uninstalled ase-3.28.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: torchani
    Found existing installation: torchani 2.7.9
    Uninstalling torchani-2.7.9:
      Successfully uninstalled torchani-2.7.9
Return code: 0
openmm-ml Return code: 1


In [61]:
# GPU-02 Retry: Re-execute test after torchani reinstall
import subprocess, sys, datetime

print(f"GPU-02 Retry Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "test_ml_potential_creates_valid_system",
     "-v", "--tb=long"],
    capture_output=True, text=True, timeout=600,
    cwd="/content"
)

# Compact output
print(f"Return code: {result.returncode}")
for line in result.stdout.splitlines():
    ls = line.strip()
    if any(kw in ls for kw in [
        "PASSED", "FAILED", "SKIPPED", "ERROR",
        "PARTICLES=", "FORCES=", "FORCE_TYPES=", "SUCCESS",
        "passed", "failed", "skipped", "test_ml_potential",
        "ImportError", "AssertionError", "AssertionError",
    ]):
        print(f"  {ls}")

if result.stderr:
    stderr_lines = result.stderr.strip().splitlines()
    # Show last 5 lines of stderr
    print(f"\nSTDERR (last 5 of {len(stderr_lines)}):")
    for line in stderr_lines[-5:]:
        print(f"  {line}")

if "1 passed" in result.stdout:
    print("\n✅ GPU-02: PASSED")
elif "PASSED" in result.stdout and result.returncode in (0, 139, -11):
    print("\n✅ GPU-02: PASSED (with segfault — expected)")
elif "1 skipped" in result.stdout:
    print("\n⏭️ GPU-02: SKIPPED")
else:
    print("\n❌ GPU-02: FAILED")

GPU-02 Retry Timestamp: 2026-03-20 22:53
Return code: 1
  tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [100%]
  ____________________ test_ml_potential_creates_valid_system ____________________
  def test_ml_potential_creates_valid_system():
  print(f"PARTICLES={n_particles}")
  print(f"FORCES={n_forces}")
  print(f"FORCE_TYPES={','.join(force_types)}")
  print("SUCCESS")
  >       assert "SUCCESS" in stdout, (
  f"ML system creation failed.\nstdout: {stdout}\nstderr: {result.stderr[-500:]}"
  E       AssertionError: ML system creation failed.
  E             raise ImportError(f"Failed to import torchani with error: {e}. Make sure torchani is installed.")
  E         ImportError: Failed to import torchani with error: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: _ZNK2at7Context14allowTF32CuDNNEv. Make sure torchani is installed.
  E       assert 'SUCCESS' in ''
  tests/test_force_field_factory.py:215: AssertionE

In [62]:
# GPU-02 Fix: Install openmm-torch from pip to match pip torch 2.8.0
# The conda openmm-torch (CXXABI_1.3.15 issue) conflicts with pip torch
import subprocess, sys

# Try pip install of openmm-torch
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "openmm-torch", "--no-cache-dir"],
    capture_output=True, text=True, timeout=300
)
lines = r.stdout.strip().splitlines()
for line in lines[-5:]:
    print(line)
if r.returncode != 0:
    print(f"pip openmm-torch failed (code {r.returncode})")
    # Show error
    for line in r.stderr.strip().splitlines()[-10:]:
        print(f"  ERR: {line}")
else:
    print("✅ openmm-torch installed from pip")

# Verify it imports in subprocess now
r2 = subprocess.run(
    [sys.executable, "-c", 
     "import openmmtorch; print('openmmtorch import OK'); "
     "from openmmml import MLPotential; p = MLPotential('ani2x'); print('MLPotential OK')"],
    capture_output=True, text=True, timeout=60
)
print(f"\nVerification stdout: {r2.stdout.strip()}")
if r2.stderr:
    stderr_lines = r2.stderr.strip().splitlines()
    print(f"Verification stderr (last 3): {stderr_lines[-3:]}")
print(f"Verification return code: {r2.returncode}")

pip openmm-torch failed (code 1)
  ERR: ERROR: Could not find a version that satisfies the requirement openmm-torch (from versions: none)
  ERR: ERROR: No matching distribution found for openmm-torch

Verification stdout: openmmtorch import OK
MLPotential OK
Verification stderr (last 3): ["Warning: importing 'simtk.openmm' is deprecated.  Import 'openmm' instead."]
Verification return code: 0


In [63]:
# GPU-02 Retry 2: Re-execute after environment verification
import subprocess, sys, datetime

print(f"GPU-02 Retry 2 Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "test_ml_potential_creates_valid_system",
     "-v", "--tb=long", "--no-header"],
    capture_output=True, text=True, timeout=600,
    cwd="/content"
)

print(f"Return code: {result.returncode}")
print()

# Show all stdout
for line in result.stdout.splitlines():
    ls = line.strip()
    if ls:
        print(f"  {ls}")

if result.stderr:
    stderr_lines = result.stderr.strip().splitlines()
    print(f"\nSTDERR (last 5 of {len(stderr_lines)}):")
    for line in stderr_lines[-5:]:
        print(f"  {line}")

if "1 passed" in result.stdout:
    print("\n✅ GPU-02: PASSED")
elif "PASSED" in result.stdout and result.returncode in (0, 139, -11):
    print("\n✅ GPU-02: PASSED (with expected cleanup segfault)")
elif "1 skipped" in result.stdout:
    print("\n⏭️ GPU-02: SKIPPED")
else:
    print("\n❌ GPU-02: FAILED")

GPU-02 Retry 2 Timestamp: 2026-03-20 22:55
Return code: 1

  ============================= test session starts ==============================
  collecting ... collected 10 items / 9 deselected / 1 selected
  tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [100%]
  =================================== FAILURES ===================================
  ____________________ test_ml_potential_creates_valid_system ____________________
  @pytest.mark.skipif(
  not _openmmml_available(),
  reason="openmmml/torchani not installed",
  )
  def test_ml_potential_creates_valid_system():
  """ANI-2x via openmmml should create a valid OpenMM System with TorchForce.
  Validates that the openmmml -> torchani -> TorchForce pipeline produces
  a well-formed System object. Runs the actual system creation in a
  subprocess to prevent the openmmtorch exit-cleanup segfault (macOS ARM
  pip install) from crashing the pytest process.
  """
  import subprocess
  import sys
  import 

In [64]:
# GPU-02 Retry 2 - Compact Result Check
print(f"Return code: {result.returncode}")
print(f"'1 passed' in stdout: {'1 passed' in result.stdout}")
print(f"'PASSED' in stdout: {'PASSED' in result.stdout}")
print(f"'FAILED' in stdout: {'FAILED' in result.stdout}")
print(f"'SUCCESS' in stdout: {'SUCCESS' in result.stdout}")
print(f"'PARTICLES=22' in stdout: {'PARTICLES=22' in result.stdout}")

# Get pass/fail line
for line in result.stdout.splitlines():
    ls = line.strip()
    if 'test_ml_potential_creates_valid_system' in ls and any(w in ls for w in ['PASSED', 'FAILED', 'SKIPPED']):
        print(f"\nResult line: {ls}")
    if ls.startswith('=') and ('passed' in ls or 'failed' in ls):
        print(f"Summary: {ls}")

Return code: 1
'1 passed' in stdout: False
'PASSED' in stdout: False
'FAILED' in stdout: True
'SUCCESS' in stdout: True
'PARTICLES=22' in stdout: False

Result line: tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [100%]

Result line: FAILED tests/test_force_field_factory.py::test_ml_potential_creates_valid_system
Summary: ======================= 1 failed, 9 deselected in 2.70s ========================


In [65]:
# GPU-02 Debug: Test exact createSystem path in subprocess
import subprocess, sys, textwrap

script = textwrap.dedent("""\
    import sys, os, traceback
    sys.path.insert(0, "/content")
    
    print(f"torch version: {__import__('torch').__version__}")
    print(f"torch path: {__import__('torch').__file__}")
    
    # Step 1: Import openmmml
    try:
        from openmmml import MLPotential
        print("step1: openmmml import OK")
    except Exception as e:
        print(f"step1 FAILED: {e}")
        sys.exit(1)
    
    # Step 2: Create MLPotential object
    try:
        potential = MLPotential("ani2x")
        print("step2: MLPotential('ani2x') OK")
    except Exception as e:
        print(f"step2 FAILED: {e}")
        traceback.print_exc()
        sys.exit(1)
    
    # Step 3: Build topology
    try:
        from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide
        topology, positions = _build_unsolvated_alanine_dipeptide()
        print(f"step3: topology built, {topology.getNumAtoms()} atoms")
    except Exception as e:
        print(f"step3 FAILED: {e}")
        traceback.print_exc()
        sys.exit(1)
    
    # Step 4: createSystem (this is where the error likely occurs)
    try:
        system = potential.createSystem(topology, implementation="torchani")
        print(f"step4: createSystem OK, particles={system.getNumParticles()}")
    except Exception as e:
        print(f"step4 FAILED: {e}")
        traceback.print_exc()
        sys.exit(1)
    
    print("ALL STEPS PASSED")
""")

r = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120, cwd="/content"
)
print("STDOUT:")
print(r.stdout)
if r.stderr:
    stderr_lines = r.stderr.strip().splitlines()
    print(f"\nSTDERR (last 10 of {len(stderr_lines)}):")
    for line in stderr_lines[-10:]:
        print(f"  {line}")
print(f"\nReturn code: {r.returncode}")

STDOUT:
torch version: 2.8.0+cu128
torch path: /usr/local/lib/python3.12/dist-packages/torch/__init__.py
step1: openmmml import OK
step2: MLPotential('ani2x') OK
step3: topology built, 22 atoms
step4: createSystem OK, particles=22
ALL STEPS PASSED


STDERR (last 10 of 3):
  /usr/local/lib/python3.12/dist-packages/torchani/csrc/__init__.py:56: UserWarning: The extensions: ['cuaev', 'mnp', 'cell_list'] are not installed and will not be available. To install the extensions first install the CUDA Toolkit, and afterwards  run `ani build-extensions` To suppress warn set the env var TORCHANI_NO_WARN_EXTENSIONS=1 For example, if using bash, you may add `export TORCHANI_NO_WARN_EXTENSIONS=1` to your .bashrc
    warnings.warn(

Return code: -11


In [66]:
# GPU-02 Retry 3: Clear __pycache__ and re-run pytest
import subprocess, sys, datetime, shutil, os

# Clear all __pycache__ directories to avoid stale bytecode
for root, dirs, files in os.walk("/content"):
    for d in dirs:
        if d == "__pycache__":
            p = os.path.join(root, d)
            shutil.rmtree(p)
            print(f"Cleared: {p}")

print(f"\nGPU-02 Retry 3 Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "test_ml_potential_creates_valid_system",
     "-v", "--tb=short", "--no-header", "-p", "no:cacheprovider"],
    capture_output=True, text=True, timeout=600,
    cwd="/content"
)

print(f"Return code: {result.returncode}")
# Extract key result lines only
for line in result.stdout.splitlines():
    ls = line.strip()
    if any(kw in ls for kw in ['test_ml_potential_creates_valid_system', 'passed', 'failed', 'skipped',
                                'PARTICLES=', 'SUCCESS', 'ImportError', 'assert']):
        print(f"  {ls}")

if "1 passed" in result.stdout:
    print("\n✅ GPU-02: PASSED")
elif "PASSED" in result.stdout and result.returncode in (0, 139, -11):
    print("\n✅ GPU-02: PASSED (with expected segfault)")
else:
    print("\n❌ GPU-02: Still failing")

Cleared: /content/tests/__pycache__
Cleared: /content/src/__pycache__
Cleared: /content/src/physics/__pycache__

GPU-02 Retry 3 Timestamp: 2026-03-20 22:56
Return code: 1
  tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [100%]
  ____________________ test_ml_potential_creates_valid_system ____________________
  tests/test_force_field_factory.py:215: in test_ml_potential_creates_valid_system
  assert "SUCCESS" in stdout, (
  E   AssertionError: ML system creation failed.
  E         raise ImportError(f"Failed to import torchani with error: {e}. Make sure torchani is installed.")
  E     ImportError: Failed to import torchani with error: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: _ZNK2at7Context14allowTF32CuDNNEv. Make sure torchani is installed.
  E   assert 'SUCCESS' in ''
  FAILED tests/test_force_field_factory.py::test_ml_potential_creates_valid_system
  ======================= 1 failed, 9 deselected in 2.71s

In [67]:
# GPU-02 Debug: Check if torch import modifies LD_LIBRARY_PATH
import subprocess, sys

# Check what happens when torch is imported in a subprocess
r = subprocess.run([sys.executable, "-c", 
    "import os; "
    "print('BEFORE LD_LIBRARY_PATH:', os.environ.get('LD_LIBRARY_PATH', '(not set)')); "
    "import torch; "
    "print('AFTER  LD_LIBRARY_PATH:', os.environ.get('LD_LIBRARY_PATH', '(not set)')); "
    "print('torch.__file__:', torch.__file__)"
], capture_output=True, text=True, timeout=30)
print(r.stdout)

# Now simulate what pytest does: import torch then spawn inner subprocess
r2 = subprocess.run([sys.executable, "-c", """
import os, sys, subprocess

# Simulate pytest: import torch (test collection triggers _openmmml_available which imports openmmml)
import torch
import openmmml

# Now spawn inner subprocess (like the test does)
inner_script = '''
import os, sys
sys.path.insert(0, "/content")
print(f"LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', '(not set)')}")
print(f"torch: {__import__('torch').__file__} version={__import__('torch').__version__}")
try:
    from openmmml import MLPotential
    p = MLPotential("ani2x")
    from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide
    topology, positions = _build_unsolvated_alanine_dipeptide()
    system = p.createSystem(topology, implementation="torchani")
    print(f"PARTICLES={system.getNumParticles()}")
    print("SUCCESS")
except Exception as e:
    print(f"FAILED: {e}")
'''

r = subprocess.run([sys.executable, "-c", inner_script],
                   capture_output=True, text=True, timeout=120, cwd="/content")
print("INNER STDOUT:", r.stdout)
if r.stderr:
    lines = r.stderr.strip().splitlines()
    print("INNER STDERR (last 3):", lines[-3:])
print("INNER RETURN CODE:", r.returncode)
"""], capture_output=True, text=True, timeout=180, cwd="/content")

print("\nSIMULATED PYTEST OUTPUT:")
print(r2.stdout)
if r2.returncode != 0:
    print(f"Return code: {r2.returncode}")
    if r2.stderr:
        lines = r2.stderr.strip().splitlines()
        print("STDERR (last 5):", lines[-5:])

BEFORE LD_LIBRARY_PATH: /opt/conda/envs/gpu/lib:/opt/conda/envs/gpu/lib:/usr/lib64-nvidia
AFTER  LD_LIBRARY_PATH: /opt/conda/envs/gpu/lib:/opt/conda/envs/gpu/lib:/usr/lib64-nvidia
torch.__file__: /usr/local/lib/python3.12/dist-packages/torch/__init__.py


SIMULATED PYTEST OUTPUT:
INNER STDOUT: LD_LIBRARY_PATH: /opt/conda/envs/gpu/lib:/opt/conda/envs/gpu/lib:/usr/lib64-nvidia
torch: /usr/local/lib/python3.12/dist-packages/torch/__init__.py version=2.8.0+cu128
PARTICLES=22
SUCCESS

INNER STDERR (last 3): ["/usr/local/lib/python3.12/dist-packages/torchani/csrc/__init__.py:56: UserWarning: The extensions: ['cuaev', 'mnp', 'cell_list'] are not installed and will not be available. To install the extensions first install the CUDA Toolkit, and afterwards  run `ani build-extensions` To suppress warn set the env var TORCHANI_NO_WARN_EXTENSIONS=1 For example, if using bash, you may add `export TORCHANI_NO_WARN_EXTENSIONS=1` to your .bashrc", '  warnings.warn(', "Warning: importing 'simtk.openmm' 

In [68]:
# GPU-02: Run test function directly (bypass pytest framework)
import subprocess, sys, datetime

print(f"GPU-02 Direct Run: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 70)

# Run the exact test function without pytest, but check all assertions
r = subprocess.run([sys.executable, "-c", """
import sys, os
sys.path.insert(0, "/content")
os.chdir("/content")

# Run the test function directly
from tests.test_force_field_factory import test_ml_potential_creates_valid_system
try:
    test_ml_potential_creates_valid_system()
    print("TEST_RESULT: PASSED")
except AssertionError as e:
    print(f"TEST_RESULT: FAILED (assertion: {e})")
except Exception as e:
    print(f"TEST_RESULT: ERROR ({type(e).__name__}: {e})")
"""], capture_output=True, text=True, timeout=600, cwd="/content")

print(f"Return code: {r.returncode}")
print("\nSTDOUT:")
for line in r.stdout.splitlines():
    print(f"  {line}")
if r.stderr:
    stderr_lines = r.stderr.strip().splitlines()
    print(f"\nSTDERR (last 5 of {len(stderr_lines)}):")
    for line in stderr_lines[-5:]:
        print(f"  {line}")

GPU-02 Direct Run: 2026-03-20 22:59
Return code: 0

STDOUT:
  TEST_RESULT: FAILED (assertion: ML system creation failed.
  stdout: 
  stderr: ", line 230, in createSystem
      self._impl.addForces(topology, system, None, 0, **args)
    File "/usr/local/lib/python3.12/dist-packages/openmmml/models/anipotential.py", line 74, in addForces
      raise ImportError(f"Failed to import torchani with error: {e}. Make sure torchani is installed.")
  ImportError: Failed to import torchani with error: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: _ZNK2at7Context14allowTF32CuDNNEv. Make sure torchani is installed.
  )


In [69]:
# GPU-02 Hypothesis: Verify import torch before openmm fixes the issue
import subprocess, sys, textwrap

script = textwrap.dedent("""\
    import sys, os
    sys.path.insert(0, os.getcwd())
    # FIX: Import torch FIRST to ensure pip's libtorch_cuda.so is loaded 
    # before OpenMM's plugin loader can pull in conda's conflicting version
    import torch
    from src.config import SystemConfig, MLPotentialConfig
    from src.physics.force_field_factory import create_system
    from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide

    topology, positions = _build_unsolvated_alanine_dipeptide()
    config = SystemConfig(force_field_family="ml_potential")
    ml_config = MLPotentialConfig(potential_name="ani2x")
    system = create_system(topology, positions, config, ml_config=ml_config)

    n_particles = system.getNumParticles()
    n_forces = system.getNumForces()
    force_types = [type(system.getForce(i)).__name__ for i in range(n_forces)]

    print(f"PARTICLES={n_particles}")
    print(f"FORCES={n_forces}")
    print(f"FORCE_TYPES={','.join(force_types)}")
    print("SUCCESS")
""")

r = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120, cwd="/content"
)
print("STDOUT:")
print(r.stdout)
print(f"Return code: {r.returncode}")
if "SUCCESS" in r.stdout and "PARTICLES=22" in r.stdout:
    print("\n✅ Hypothesis confirmed: import torch before openmm fixes the issue")

STDOUT:
PARTICLES=22
FORCES=2
FORCE_TYPES=Force,CMMotionRemover
SUCCESS

Return code: -11

✅ Hypothesis confirmed: import torch before openmm fixes the issue


In [70]:
# GPU-02 Fix: Patch test file on Colab to import torch before openmm in subprocess
# This prevents conda's libtorch_cuda.so from being loaded first by OpenMM's plugin loader
import os

test_file = "/content/tests/test_force_field_factory.py"
with open(test_file, "r") as f:
    content = f.read()

# Apply the fix: add 'import torch' before the openmm-importing lines in the subprocess script
old = '''        import sys, os
        sys.path.insert(0, os.getcwd())
        from src.config import SystemConfig, MLPotentialConfig
        from src.physics.force_field_factory import create_system
        from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide'''

new = '''        import sys, os
        sys.path.insert(0, os.getcwd())
        import torch  # must precede openmm to avoid libtorch_cuda.so symbol conflicts
        from src.config import SystemConfig, MLPotentialConfig
        from src.physics.force_field_factory import create_system
        from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide'''

if old in content:
    content = content.replace(old, new, 1)
    with open(test_file, "w") as f:
        f.write(content)
    print("✅ Test file patched successfully on Colab runtime")
    # Verify the fix is in place
    with open(test_file) as f:
        for i, line in enumerate(f, 1):
            if "import torch" in line and "must precede openmm" in line:
                print(f"   Fix at line {i}: {line.strip()}")
elif "import torch  # must precede openmm" in content:
    print("✅ Fix already applied")
else:
    print("❌ Could not find the target string to patch")

✅ Test file patched successfully on Colab runtime
   Fix at line 187: import torch  # must precede openmm to avoid libtorch_cuda.so symbol conflicts


In [71]:
# GPU-02 FINAL: Execute via pytest with patched test file
import subprocess, sys, datetime

timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
print(f"GPU-02 Final Execution: {timestamp}")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "test_ml_potential_creates_valid_system",
     "-v", "--tb=short", "-p", "no:cacheprovider"],
    capture_output=True, text=True, timeout=600,
    cwd="/content"
)

print(f"Return code: {result.returncode}")

# Extract key lines
for line in result.stdout.splitlines():
    ls = line.strip()
    if any(kw in ls for kw in [
        'test_ml_potential_creates_valid_system', 
        'passed', 'failed', 'skipped',
        'PARTICLES=', 'FORCES=', 'FORCE_TYPES=', 'SUCCESS',
    ]):
        print(f"  {ls}")

if "1 passed" in result.stdout:
    print(f"\n✅ GPU-02: PASSED at {timestamp}")
elif "PASSED" in result.stdout:
    print(f"\n✅ GPU-02: PASSED at {timestamp}")
elif "1 skipped" in result.stdout:
    print(f"\n⏭️ GPU-02: SKIPPED at {timestamp}")
else:
    print(f"\n❌ GPU-02: FAILED at {timestamp}")
    # Show error details
    for line in result.stdout.splitlines():
        ls = line.strip()
        if 'Error' in ls or 'assert' in ls.lower():
            print(f"  ERROR: {ls}")

GPU-02 Final Execution: 2026-03-20 23:03
Return code: 0
  tests/test_force_field_factory.py::test_ml_potential_creates_valid_system PASSED [100%]
  ======================= 1 passed, 9 deselected in 6.50s ========================

✅ GPU-02: PASSED at 2026-03-20 23:03


In [72]:
# GPU-02 Output Details: Extract subprocess output from within the test
# Re-run the fixed subprocess script directly to capture full output
import subprocess, sys, textwrap

script = textwrap.dedent("""\
    import sys, os
    sys.path.insert(0, os.getcwd())
    import torch
    from src.config import SystemConfig, MLPotentialConfig
    from src.physics.force_field_factory import create_system
    from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide

    topology, positions = _build_unsolvated_alanine_dipeptide()
    config = SystemConfig(force_field_family="ml_potential")
    ml_config = MLPotentialConfig(potential_name="ani2x")
    system = create_system(topology, positions, config, ml_config=ml_config)

    n_particles = system.getNumParticles()
    n_forces = system.getNumForces()
    force_types = [type(system.getForce(i)).__name__ for i in range(n_forces)]

    print(f"PARTICLES={n_particles}")
    print(f"FORCES={n_forces}")
    print(f"FORCE_TYPES={','.join(force_types)}")
    print(f"TORCH_VERSION={torch.__version__}")
    print(f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
    print("SUCCESS")
""")

r = subprocess.run([sys.executable, "-c", script],
                   capture_output=True, text=True, timeout=120, cwd="/content")
print("Subprocess output:")
print(r.stdout)
print(f"Return code: {r.returncode} {'(expected segfault on exit)' if r.returncode == -11 else ''}")

Subprocess output:
PARTICLES=22
FORCES=2
FORCE_TYPES=Force,CMMotionRemover
TORCH_VERSION=2.8.0+cu128
GPU=NVIDIA A100-SXM4-80GB
SUCCESS

Return code: -11 (expected segfault on exit)


In [73]:
# GPU-03: ANI-2x Direct Energy Evaluation via TorchANI
# Run test_ml_potential_ani2x_direct_energy via pytest
import subprocess, sys, datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 execution started: {timestamp}")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-xvs",
     "tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy"],
    capture_output=True, text=True, timeout=300, cwd="/content"
)

print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR (last 30 lines):")
    print("\n".join(result.stderr.strip().split("\n")[-30:]))
print(f"\nReturn code: {result.returncode}")
print(f"GPU-03 result: {'PASSED' if result.returncode == 0 else 'FAILED/ERROR'}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 execution completed: {timestamp_end}")

GPU-03 execution started: 2026-03-20 23:16
STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.12.1, langsmith-0.7.18, typeguard-4.5.1
collecting ... collected 1 item

tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy FAILED

=================================== FAILURES ===================================
____________________ test_ml_potential_ani2x_direct_energy _____________________

    @pytest.mark.skipif(
        not _openmmml_available(),
        reason="openmmml/torchani not installed",
    )
    def test_ml_potential_ani2x_direct_energy():
        """ANI-2x should compute finite energies for alanine dipeptide via torchani directly.
    
        Bypasses the OpenMM Context/TorchForce kernel to validate ANI-2x energy
        evaluation independently. This confirms the neural network 

In [74]:
# GPU-03 Result Summary (compact)
# Re-check the result from the previous cell
print("=== GPU-03 RESULT SUMMARY ===")
print(f"Return code: {result.returncode}")

# Extract key lines from stdout
stdout_lines = result.stdout.strip().split("\n") if result.stdout else []
for line in stdout_lines:
    if any(k in line.upper() for k in ["PASSED", "FAILED", "ERROR", "SKIP", "ANI", "ENERGY", "FINITE", "NEGATIVE"]):
        print(line)

# Show last 15 lines of stdout for pytest summary
print("\n--- Last 15 lines of stdout ---")
for line in stdout_lines[-15:]:
    print(line)

# Show last 5 lines of stderr if any
if result.stderr:
    stderr_lines = result.stderr.strip().split("\n")
    print("\n--- Last 5 lines of stderr ---")
    for line in stderr_lines[-5:]:
        print(line)

print(f"\nGPU-03 VERDICT: {'PASSED' if result.returncode == 0 else 'FAILED/ERROR'}")

=== GPU-03 RESULT SUMMARY ===
Return code: 1
tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy FAILED
____________________ test_ml_potential_ani2x_direct_energy _____________________
    @pytest.mark.skipif(
        reason="openmmml/torchani not installed",
    def test_ml_potential_ani2x_direct_energy():
        """ANI-2x should compute finite energies for alanine dipeptide via torchani directly.
        Bypasses the OpenMM Context/TorchForce kernel to validate ANI-2x energy
        produces physically reasonable energies for organic molecules.
    except ImportError:
            kernel32 = ctypes.WinDLL("kernel32.dll", use_last_error=True)
            prev_error_mode = kernel32.SetErrorMode(0x0001)
            except OSError:
                    last_error = ctypes.get_last_error()
                    if res is None and last_error != 126:
                        err = ctypes.WinError(last_error)
                        err.strerror += (
                        

In [75]:
# GPU-03 Minimal Result
print(f"rc={result.returncode}")
lines = result.stdout.strip().split("\n") if result.stdout else []
print(f"total_lines={len(lines)}")
# Print only last 5 lines
for l in lines[-5:]:
    print(l)

rc=1
total_lines=452
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:416: ImportError
=========================== short test summary info ============================
FAILED tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy
!!!!!!!!!!!!!!!!!!!!!!!!!! stopping after 1 failures !!!!!!!!!!!!!!!!!!!!!!!!!!!
============================== 1 failed in 0.84s ===============================


In [76]:
# GPU-03 Error Details
lines = result.stdout.strip().split("\n") if result.stdout else []
# Find FAILED/ERROR/assert lines and context
for i, l in enumerate(lines):
    if any(k in l for k in ["FAILED", "Error", "assert", "raise", "energy", "ENERGY", "ImportError", "torchani"]):
        # Print surrounding context
        start = max(0, i-3)
        end = min(len(lines), i+4)
        for j in range(start, end):
            print(f"[{j}] {lines[j]}")
        print("---")

[4] plugins: anyio-4.12.1, langsmith-0.7.18, typeguard-4.5.1
[5] collecting ... collected 1 item
[6] 
[7] tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy FAILED
[8] 
[9] =================================== FAILURES ===================================
[10] ____________________ test_ml_potential_ani2x_direct_energy _____________________
---
[7] tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy FAILED
[8] 
[9] =================================== FAILURES ===================================
[10] ____________________ test_ml_potential_ani2x_direct_energy _____________________
[11] 
[12]     @pytest.mark.skipif(
[13]         not _openmmml_available(),
---
[11] 
[12]     @pytest.mark.skipif(
[13]         not _openmmml_available(),
[14]         reason="openmmml/torchani not installed",
[15]     )
[16]     def test_ml_potential_ani2x_direct_energy():
[17]         """ANI-2x should compute finite energies for alanine dipeptide via torchani directly.

In [77]:
# GPU-03 Focused Error - print lines 430-452 only (end of output)
lines = result.stdout.strip().split("\n")
for i in range(max(0,len(lines)-25), len(lines)):
    print(f"[{i}] {lines[i]}")

[427]     
[428]         sys.setdlopenflags(old_flags)
[429]         del old_flags
[430]     
[431]     else:
[432]         # Easy way.  You want this most of the time, because it will prevent
[433]         # C++ symbols from libtorch clobbering C++ symbols from other
[434]         # libraries, leading to mysterious segfaults.
[435]         #
[436]         # If building in an environment where libtorch_global_deps isn't available
[437]         # like parts of fbsource, but where RTLD_GLOBAL causes segfaults, you will
[438]         # want USE_RTLD_GLOBAL_WITH_LIBTORCH = False and USE_GLOBAL_DEPS = False
[439]         #
[440]         # See Note [Global dependencies]
[441]         if USE_GLOBAL_DEPS:
[442]             _load_global_deps()
[443] >       from torch._C import *  # noqa: F403
[444]         ^^^^^^^^^^^^^^^^^^^^^^
[445] E       ImportError: /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cuda.so: undefined symbol: _ZNK2at7Context14allowTF32CuDNNEv
[446] 
[447] /usr/lo

In [78]:
# GPU-03: Direct execution in notebook kernel (no subprocess needed)
# test_ml_potential_ani2x_direct_energy - ANI-2x direct energy via TorchANI
import datetime, numpy as np, torch, torchani
from openmm import unit
from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 execution started: {timestamp_start}")
print("=" * 60)

# Build the alanine dipeptide topology and positions
topology, positions = _build_unsolvated_alanine_dipeptide()

# Extract atomic numbers and coordinates
elements_to_z = {"hydrogen": 1, "carbon": 6, "nitrogen": 7, "oxygen": 8}
species = []
coords = []
for atom in topology.atoms():
    z = elements_to_z.get(atom.element.name.lower())
    if z is None:
        raise RuntimeError(f"Unsupported element for ANI-2x: {atom.element.name}")
    species.append(z)

for i in range(len(species)):
    pos = positions[i]
    coords.append([
        pos[0].value_in_unit(unit.angstrom),
        pos[1].value_in_unit(unit.angstrom),
        pos[2].value_in_unit(unit.angstrom),
    ])

species_tensor = torch.tensor([species], dtype=torch.long)
coords_tensor = torch.tensor([coords], dtype=torch.float32, requires_grad=True)

print(f"Atoms: {len(species)}")
print(f"Species tensor shape: {species_tensor.shape}")
print(f"Coords tensor shape: {coords_tensor.shape}")
print(f"Atomic numbers: {species}")

# Load ANI-2x model and compute energy
model = torchani.models.ANI2x(periodic_table_index=True)
result = model((species_tensor, coords_tensor))
energy = result.energies
energy_val = energy.item()

print(f"\nANI-2x Energy: {energy_val} Hartrees")
print(f"np.isfinite(energy): {np.isfinite(energy_val)}")
print(f"energy < 0: {energy_val < 0}")

# Assertions matching the test
assert np.isfinite(energy_val), f"ANI-2x energy must be finite, got {energy_val}"
assert energy_val < 0, f"ANI-2x energy should be negative for a stable molecule, got {energy_val}"

print(f"\nTorchANI version: {torchani.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"\n{'=' * 60}")
print(f"GPU-03 RESULT: PASSED")
print(f"GPU-03 execution completed: {timestamp_end}")

GPU-03 execution started: 2026-03-20 23:18


: 

: 

: 

In [1]:
# Recover environment after kernel restart
import os, sys
sys.path.insert(0, "/content")
os.chdir("/content")
print(f"cwd: {os.getcwd()}")
print(f"Python: {sys.executable}")

cwd: /content
Python: /usr/bin/python3


In [2]:
# Check current env for library paths
import os
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH", "NOT SET"))
print("PYTHONPATH:", os.environ.get("PYTHONPATH", "NOT SET"))

# Check if torch imports fine in current kernel
try:
    import torch
    print(f"torch OK: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
except Exception as e:
    print(f"torch FAILED: {e}")

try:
    import torchani
    print(f"torchani OK: {torchani.__version__}")
except Exception as e:
    print(f"torchani FAILED: {e}")

LD_LIBRARY_PATH: /usr/lib64-nvidia
PYTHONPATH: /env/python
torch OK: 2.8.0+cu128, CUDA: True


/usr/local/lib/python3.12/dist-packages/torchani/csrc/__init__.py:56: UserWarning: The extensions: ['cuaev', 'mnp', 'cell_list'] are not installed and will not be available. To install the extensions first install the CUDA Toolkit, and afterwards  run `ani build-extensions` To suppress warn set the env var TORCHANI_NO_WARN_EXTENSIONS=1 For example, if using bash, you may add `export TORCHANI_NO_WARN_EXTENSIONS=1` to your .bashrc
  warnings.warn(


torchani OK: 2.7.9


In [3]:
# GPU-03: Run via subprocess with self-contained script
# Avoids importing openmm-torch (which causes segfault on cleanup)
import subprocess, sys, os, datetime, textwrap

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 execution started: {timestamp_start}")

script = textwrap.dedent("""\
    import sys, os, numpy as np
    sys.path.insert(0, "/content")
    os.chdir("/content")
    
    import torch
    import torchani
    from openmm import unit
    
    # Replicate _build_unsolvated_alanine_dipeptide inline to avoid
    # importing the full test module (which triggers openmm-torch)
    from openmm.app import ForceField, Modeller, Topology
    from openmm.app.element import carbon, nitrogen, oxygen, hydrogen
    import openmm.app as app
    
    # Build ACE-ALA-NME (alanine dipeptide) topology
    fixer_pdb = os.path.join("/content", "pdbs", "alanine_dipeptide.pdb")
    if os.path.exists(fixer_pdb):
        from openmm.app import PDBFile
        pdb = PDBFile(fixer_pdb)
        topology = pdb.topology
        positions = pdb.positions
    else:
        # Fallback: build from AMBER ff14SB
        from pdbfixer import PDBFixer
        import io
        # Use Modeller approach from the test helper
        from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide
        topology, positions = _build_unsolvated_alanine_dipeptide()
    
    # Extract atomic numbers and coordinates
    elements_to_z = {"hydrogen": 1, "carbon": 6, "nitrogen": 7, "oxygen": 8}
    species = []
    coords = []
    for atom in topology.atoms():
        z = elements_to_z.get(atom.element.name.lower())
        if z is None:
            print(f"SKIP: Unsupported element {atom.element.name}")
            sys.exit(2)
        species.append(z)
    
    for i in range(len(species)):
        pos = positions[i]
        coords.append([
            pos[0].value_in_unit(unit.angstrom),
            pos[1].value_in_unit(unit.angstrom),
            pos[2].value_in_unit(unit.angstrom),
        ])
    
    species_tensor = torch.tensor([species], dtype=torch.long)
    coords_tensor = torch.tensor([coords], dtype=torch.float32, requires_grad=True)
    
    print(f"ATOMS={len(species)}")
    print(f"SPECIES={species}")
    
    model = torchani.models.ANI2x(periodic_table_index=True)
    result = model((species_tensor, coords_tensor))
    energy = result.energies
    energy_val = energy.item()
    
    print(f"ENERGY={energy_val}")
    print(f"FINITE={np.isfinite(energy_val)}")
    print(f"NEGATIVE={energy_val < 0}")
    print(f"TORCHANI_VERSION={torchani.__version__}")
    print(f"TORCH_VERSION={torch.__version__}")
    print(f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
    
    assert np.isfinite(energy_val), f"Energy not finite: {energy_val}"
    assert energy_val < 0, f"Energy not negative: {energy_val}"
    
    print("SUCCESS")
""")

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "/usr/lib64-nvidia")

r = subprocess.run([sys.executable, "-c", script],
                   capture_output=True, text=True, timeout=120,
                   cwd="/content", env=env)

print("STDOUT:")
print(r.stdout)
if r.stderr:
    stderr_lines = r.stderr.strip().split("\n")
    # Filter out torchani extension warnings
    important = [l for l in stderr_lines if "WARN_EXTENSIONS" not in l and "UserWarning" not in l and "warnings.warn" not in l]
    if important:
        print("STDERR (filtered):")
        for l in important[-10:]:
            print(l)
print(f"\nReturn code: {r.returncode}")
has_success = "SUCCESS" in r.stdout
print(f"GPU-03 RESULT: {'PASSED' if (r.returncode == 0 or has_success) else 'FAILED'}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 completed: {timestamp_end}")

GPU-03 execution started: 2026-03-20 23:19
STDOUT:

STDERR (filtered):
Traceback (most recent call last):
  File "<string>", line 24, in <module>
ModuleNotFoundError: No module named 'pdbfixer'

Return code: 1
GPU-03 RESULT: FAILED
GPU-03 completed: 2026-03-20 23:19


In [4]:
# GPU-03: Fixed subprocess - inline the topology builder
import subprocess, sys, os, datetime, textwrap

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 execution started: {timestamp_start}")

script = textwrap.dedent("""\
    import sys, os, numpy as np, tempfile
    sys.path.insert(0, "/content")
    os.chdir("/content")
    
    import torch
    import torchani
    from openmm import unit
    from openmm.app import PDBFile, Modeller, ForceField
    
    # Inline: _build_unsolvated_alanine_dipeptide
    pdb_content = '''\\
ATOM      1  CH3 ACE A   1      -2.300   1.200   0.000  1.00 20.00           C
ATOM      2  C   ACE A   1      -1.000   0.800   0.000  1.00 20.00           C
ATOM      3  O   ACE A   1      -0.300   1.700   0.000  1.00 20.00           O
ATOM      4  N   ALA A   2      -0.650  -0.500   0.000  1.00 20.00           N
ATOM      5  CA  ALA A   2       0.700  -1.050   0.000  1.00 20.00           C
ATOM      6  C   ALA A   2       1.850  -0.050   0.000  1.00 20.00           C
ATOM      7  O   ALA A   2       2.980  -0.430   0.000  1.00 20.00           O
ATOM      8  CB  ALA A   2       0.820  -2.570   0.000  1.00 20.00           C
ATOM      9  N   NME A   3       1.560   1.210   0.000  1.00 20.00           N
ATOM     10  CH3 NME A   3       2.540   2.220   0.000  1.00 20.00           C
TER
END
'''
    fd, path = tempfile.mkstemp(suffix=".pdb")
    os.write(fd, pdb_content.encode())
    os.close(fd)
    pdb = PDBFile(path)
    amber_ff = ForceField("amber14-all.xml")
    modeller = Modeller(pdb.topology, pdb.positions)
    modeller.addHydrogens(amber_ff)
    topology = modeller.topology
    positions = modeller.positions
    os.unlink(path)
    
    # Extract atomic numbers and coordinates
    elements_to_z = {"hydrogen": 1, "carbon": 6, "nitrogen": 7, "oxygen": 8}
    species = []
    coords = []
    for atom in topology.atoms():
        z = elements_to_z.get(atom.element.name.lower())
        if z is None:
            print(f"SKIP: Unsupported element {atom.element.name}")
            sys.exit(2)
        species.append(z)
    
    for i in range(len(species)):
        pos = positions[i]
        coords.append([
            pos[0].value_in_unit(unit.angstrom),
            pos[1].value_in_unit(unit.angstrom),
            pos[2].value_in_unit(unit.angstrom),
        ])
    
    species_tensor = torch.tensor([species], dtype=torch.long)
    coords_tensor = torch.tensor([coords], dtype=torch.float32, requires_grad=True)
    
    print(f"ATOMS={len(species)}")
    
    model = torchani.models.ANI2x(periodic_table_index=True)
    result = model((species_tensor, coords_tensor))
    energy = result.energies
    energy_val = energy.item()
    
    print(f"ENERGY_HARTREES={energy_val}")
    print(f"FINITE={np.isfinite(energy_val)}")
    print(f"NEGATIVE={energy_val < 0}")
    print(f"TORCHANI={torchani.__version__}")
    print(f"TORCH={torch.__version__}")
    print(f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
    
    assert np.isfinite(energy_val), f"Energy not finite: {energy_val}"
    assert energy_val < 0, f"Energy not negative: {energy_val}"
    
    print("SUCCESS")
""")

env = os.environ.copy()
r = subprocess.run([sys.executable, "-c", script],
                   capture_output=True, text=True, timeout=120,
                   cwd="/content", env=env)

print("STDOUT:")
print(r.stdout)
if r.stderr:
    stderr_lines = r.stderr.strip().split("\n")
    important = [l for l in stderr_lines 
                 if "WARN_EXTENSIONS" not in l and "UserWarning" not in l 
                 and "warnings.warn" not in l and l.strip()]
    if important:
        print("STDERR (important):")
        for l in important[-10:]:
            print(l)
print(f"\nReturn code: {r.returncode}")
has_success = "SUCCESS" in r.stdout
print(f"GPU-03 RESULT: {'PASSED' if (r.returncode == 0 or has_success) else 'FAILED'}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 completed: {timestamp_end}")

GPU-03 execution started: 2026-03-20 23:19
STDOUT:

STDERR (important):
File "<string>", line 1
    import sys, os, numpy as np, tempfile
IndentationError: unexpected indent

Return code: 1
GPU-03 RESULT: FAILED
GPU-03 completed: 2026-03-20 23:19


In [5]:
# GPU-03: Write script to file then execute
import subprocess, sys, os, datetime

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 execution started: {timestamp_start}")

script_path = "/tmp/gpu03_test.py"
with open(script_path, "w") as f:
    f.write('''import sys, os, numpy as np, tempfile
sys.path.insert(0, "/content")
os.chdir("/content")

import torch
import torchani
from openmm import unit
from openmm.app import PDBFile, Modeller, ForceField

pdb_content = """ATOM      1  CH3 ACE A   1      -2.300   1.200   0.000  1.00 20.00           C
ATOM      2  C   ACE A   1      -1.000   0.800   0.000  1.00 20.00           C
ATOM      3  O   ACE A   1      -0.300   1.700   0.000  1.00 20.00           O
ATOM      4  N   ALA A   2      -0.650  -0.500   0.000  1.00 20.00           N
ATOM      5  CA  ALA A   2       0.700  -1.050   0.000  1.00 20.00           C
ATOM      6  C   ALA A   2       1.850  -0.050   0.000  1.00 20.00           C
ATOM      7  O   ALA A   2       2.980  -0.430   0.000  1.00 20.00           O
ATOM      8  CB  ALA A   2       0.820  -2.570   0.000  1.00 20.00           C
ATOM      9  N   NME A   3       1.560   1.210   0.000  1.00 20.00           N
ATOM     10  CH3 NME A   3       2.540   2.220   0.000  1.00 20.00           C
TER
END
"""
fd, path = tempfile.mkstemp(suffix=".pdb")
os.write(fd, pdb_content.encode())
os.close(fd)
pdb = PDBFile(path)
amber_ff = ForceField("amber14-all.xml")
modeller = Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens(amber_ff)
topology = modeller.topology
positions = modeller.positions
os.unlink(path)

elements_to_z = {"hydrogen": 1, "carbon": 6, "nitrogen": 7, "oxygen": 8}
species = []
coords = []
for atom in topology.atoms():
    z = elements_to_z.get(atom.element.name.lower())
    if z is None:
        print(f"SKIP: Unsupported element {atom.element.name}")
        sys.exit(2)
    species.append(z)

for i in range(len(species)):
    pos = positions[i]
    coords.append([
        pos[0].value_in_unit(unit.angstrom),
        pos[1].value_in_unit(unit.angstrom),
        pos[2].value_in_unit(unit.angstrom),
    ])

species_tensor = torch.tensor([species], dtype=torch.long)
coords_tensor = torch.tensor([coords], dtype=torch.float32, requires_grad=True)

print(f"ATOMS={len(species)}")

model = torchani.models.ANI2x(periodic_table_index=True)
result_out = model((species_tensor, coords_tensor))
energy = result_out.energies
energy_val = energy.item()

print(f"ENERGY_HARTREES={energy_val}")
print(f"FINITE={np.isfinite(energy_val)}")
print(f"NEGATIVE={energy_val < 0}")
print(f"TORCHANI={torchani.__version__}")
print(f"TORCH={torch.__version__}")
print(f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

assert np.isfinite(energy_val), f"Energy not finite: {energy_val}"
assert energy_val < 0, f"Energy not negative: {energy_val}"

print("SUCCESS")
''')

env = os.environ.copy()
r = subprocess.run([sys.executable, script_path],
                   capture_output=True, text=True, timeout=120,
                   cwd="/content", env=env)

print("STDOUT:")
print(r.stdout)
if r.stderr:
    stderr_lines = r.stderr.strip().split("\n")
    important = [l for l in stderr_lines 
                 if "WARN_EXTENSIONS" not in l and "UserWarning" not in l 
                 and "warnings.warn" not in l and l.strip()]
    if important:
        print("STDERR (important):")
        for l in important[-10:]:
            print(l)
print(f"\nReturn code: {r.returncode}")
has_success = "SUCCESS" in r.stdout
print(f"GPU-03 RESULT: {'PASSED' if (r.returncode == 0 or has_success) else 'FAILED'}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"GPU-03 completed: {timestamp_end}")

GPU-03 execution started: 2026-03-20 23:20
STDOUT:
ATOMS=22
ENERGY_HARTREES=-495.5144958496094
FINITE=True
NEGATIVE=True
TORCHANI=2.7.9
TORCH=2.8.0+cu128
GPU=NVIDIA A100-SXM4-80GB
SUCCESS


Return code: 0
GPU-03 RESULT: PASSED
GPU-03 completed: 2026-03-20 23:20


In [6]:
# ============================================================================
# STEP 6: Combined GPU Test Run
# Run all GPU-dependent tests in a single pytest invocation
# Confirms GPU-01, GPU-02, GPU-03 pass together without interference
# ============================================================================
import subprocess, sys, os, datetime

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Step 6 — Combined GPU Test Run started: {timestamp_start}")
print("=" * 70)

env = os.environ.copy()
r = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "amoeba or ml_potential",
     "-v", "--tb=long"],
    capture_output=True, text=True, timeout=300,
    cwd="/content", env=env
)

print("STDOUT:")
print(r.stdout)
if r.stderr:
    stderr_lines = r.stderr.strip().split("\n")
    important = [l for l in stderr_lines
                 if "WARN_EXTENSIONS" not in l and "UserWarning" not in l
                 and "warnings.warn" not in l and l.strip()]
    if important:
        print("\nSTDERR (filtered):")
        for l in important[-15:]:
            print(l)

print(f"\nReturn code: {r.returncode}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Step 6 completed: {timestamp_end}")
print("=" * 70)

# Parse summary line
for line in r.stdout.split("\n"):
    if "passed" in line or "failed" in line or "error" in line:
        if "===" in line:
            print(f"\nSUMMARY: {line.strip()}")
            break

Step 6 — Combined GPU Test Run started: 2026-03-20 23:25
STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.12.1, langsmith-0.7.18, typeguard-4.5.1
collecting ... collected 10 items / 4 deselected / 6 selected

tests/test_force_field_factory.py::test_create_system_amoeba_requires_config PASSED [ 16%]
tests/test_force_field_factory.py::test_amoeba_system_produces_finite_energy PASSED [ 33%]
tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [ 50%]
tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy PASSED [ 66%]
tests/test_force_field_factory.py::test_ml_potential_produces_finite_energy SKIPPED [ 83%]
tests/test_force_field_factory.py::test_ml_potential_import_error_message SKIPPED [100%]

=================================== FAILURES ============================

In [7]:
# Step 6 Results Summary (extract from previous cell's output)
stdout_lines = r.stdout.strip().split("\n")
print("=== STEP 6 SUMMARY ===\n")
# Show test result lines
for line in stdout_lines:
    if "PASSED" in line or "FAILED" in line or "SKIPPED" in line or "===" in line:
        print(line)
print(f"\nReturn code: {r.returncode}")

=== STEP 6 SUMMARY ===

============================= test session starts ==============================
tests/test_force_field_factory.py::test_create_system_amoeba_requires_config PASSED [ 16%]
tests/test_force_field_factory.py::test_amoeba_system_produces_finite_energy PASSED [ 33%]
tests/test_force_field_factory.py::test_ml_potential_creates_valid_system FAILED [ 50%]
tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy PASSED [ 66%]
tests/test_force_field_factory.py::test_ml_potential_produces_finite_energy SKIPPED [ 83%]
tests/test_force_field_factory.py::test_ml_potential_import_error_message SKIPPED [100%]
=================================== FAILURES ===================================
=============================== warnings summary ===============================
=========================== short test summary info ============================
FAILED tests/test_force_field_factory.py::test_ml_potential_creates_valid_system
======= 1 failed, 3 passed, 2 skip

In [8]:
# Step 6 — Extract GPU-02 failure details
stdout_lines = r.stdout.strip().split("\n")
# Find FAILURES section
in_failure = False
for line in stdout_lines:
    if "FAILURES" in line and "===" in line:
        in_failure = True
    if in_failure:
        print(line)
    if in_failure and "short test summary" in line:
        break

=================================== FAILURES ===================================
____________________ test_ml_potential_creates_valid_system ____________________

    @pytest.mark.skipif(
        not _openmmml_available(),
        reason="openmmml/torchani not installed",
    )
    def test_ml_potential_creates_valid_system():
        """ANI-2x via openmmml should create a valid OpenMM System with TorchForce.
    
        Validates that the openmmml -> torchani -> TorchForce pipeline produces
        a well-formed System object. Runs the actual system creation in a
        subprocess to prevent the openmmtorch exit-cleanup segfault (macOS ARM
        pip install) from crashing the pytest process.
        """
        import subprocess
        import sys
        import textwrap
    
        script = textwrap.dedent("""\
            import sys, os
            sys.path.insert(0, os.getcwd())
            import torch  # must precede openmm to avoid libtorch_cuda.so symbol conflicts
        

In [9]:
# Step 6 — GPU-02 failure: condensed extraction
stdout_lines = r.stdout.strip().split("\n")
# Just get the lines around the failure reason
in_failure = False
failure_lines = []
for line in stdout_lines:
    if "FAILURES" in line and "===" in line:
        in_failure = True
    if in_failure:
        failure_lines.append(line)
    if in_failure and "warnings summary" in line:
        in_failure = False
        break

# Print only lines with assert, Error, FAILED, or key diagnostic info
print("GPU-02 FAILURE DIAGNOSIS (key lines):")
for line in failure_lines:
    lowline = line.lower()
    if any(kw in lowline for kw in ["assert", "error", "failed", "subprocess", "timeout", "returncode", "success", "particle", "torchforce", "===", "___"]):
        print(line)

GPU-02 FAILURE DIAGNOSIS (key lines):
=================================== FAILURES ===================================
____________________ test_ml_potential_creates_valid_system ____________________
        """ANI-2x via openmmml should create a valid OpenMM System with TorchForce.
        Validates that the openmmml -> torchani -> TorchForce pipeline produces
        subprocess to prevent the openmmtorch exit-cleanup segfault (macOS ARM
        import subprocess
            n_particles = system.getNumParticles()
            print(f"PARTICLES={n_particles}")
            print("SUCCESS")
        result = subprocess.run(
            timeout=300,
>       assert "SUCCESS" in stdout, (
            f"ML system creation failed.\nstdout: {stdout}\nstderr: {result.stderr[-500:]}"
E       AssertionError: ML system creation failed.
E         ModuleNotFoundError: No module named 'openmmtorch'
E       assert 'SUCCESS' in ''
tests/test_force_field_factory.py:216: AssertionError
====================

In [10]:
# Diagnose openmmtorch availability
import subprocess, sys
r_diag = subprocess.run([sys.executable, "-c", 
    "try:\n    import openmmtorch; print(f'openmmtorch OK: {openmmtorch.__version__}')\nexcept Exception as e:\n    print(f'openmmtorch FAIL: {e}')"],
    capture_output=True, text=True, timeout=30, cwd="/content")
print(r_diag.stdout.strip())
if r_diag.stderr.strip():
    print("stderr:", r_diag.stderr.strip()[-200:])

openmmtorch FAIL: No module named 'openmmtorch'


In [11]:
# Install openmm-torch (provides openmmtorch module needed by GPU-02 pytest test)
# openmm-ml's MLPotential.createSystem() requires this to create TorchForce objects
!pip install openmm-torch 2>&1 | tail -10

ERROR: Could not find a version that satisfies the requirement openmm-torch (from versions: none)
ERROR: No matching distribution found for openmm-torch


In [12]:
# Try conda-forge for openmm-torch
!conda install -c conda-forge openmm-torch -y 2>&1 | tail -15

/bin/bash: line 1: conda: command not found


In [13]:
# Check: 1) Is test file patched? 2) Is openmmtorch in openmm-ml?
import subprocess, sys

# Check test file on Colab
r1 = subprocess.run([sys.executable, "-c",
    "lines = open('/content/tests/test_force_field_factory.py').readlines();"
    "[print(f'{i+1}: {l.rstrip()}') for i,l in enumerate(lines[185:195])]"],
    capture_output=True, text=True, timeout=10, cwd="/content")
print("Test file lines 186-196 (subprocess script):")
print(r1.stdout)

# Check if openmmtorch is part of openmm-ml package
r2 = subprocess.run([sys.executable, "-c",
    "import openmmml, os; d=os.path.dirname(openmmml.__file__); "
    "import glob; files=glob.glob(d+'/**/*torch*', recursive=True); "
    "print(f'openmmml dir: {d}'); [print(f) for f in files]; "
    "print(f'package files: {len(files)}')"],
    capture_output=True, text=True, timeout=10, cwd="/content")
print("openmmml torch references:")
print(r2.stdout)

# Check if openmmtorch is bundled elsewhere
r3 = subprocess.run([sys.executable, "-c",
    "import importlib.util; spec=importlib.util.find_spec('openmmtorch'); "
    "print(f'openmmtorch spec: {spec}')"],
    capture_output=True, text=True, timeout=10, cwd="/content")
print("openmmtorch module location:")
print(r3.stdout)

Test file lines 186-196 (subprocess script):
1:         sys.path.insert(0, os.getcwd())
2:         import torch  # must precede openmm to avoid libtorch_cuda.so symbol conflicts
3:         from src.config import SystemConfig, MLPotentialConfig
4:         from src.physics.force_field_factory import create_system
5:         from tests.test_force_field_factory import _build_unsolvated_alanine_dipeptide
6: 
7:         topology, positions = _build_unsolvated_alanine_dipeptide()
8:         config = SystemConfig(force_field_family="ml_potential")
9:         ml_config = MLPotentialConfig(potential_name="ani2x")
10:         system = create_system(topology, positions, config, ml_config=ml_config)

openmmml torch references:
openmmml dir: /usr/local/lib/python3.12/dist-packages/openmmml
/usr/local/lib/python3.12/dist-packages/openmmml/models/torchmdnetpotential.py
/usr/local/lib/python3.12/dist-packages/openmmml/models/__pycache__/torchmdnetpotential.cpython-312.pyc
package files: 2

openmmtorch 

In [14]:
# Try alternative openmm-torch install methods
# openmm-torch provides the 'openmmtorch' Python module, needed by MLPotential.createSystem()
import subprocess, sys

# Try pip with different package names
for pkg in ["openmmtorch", "OpenMM-Torch", "openmm_torch"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"],
                       capture_output=True, text=True, timeout=60)
    if r.returncode == 0:
        print(f"✅ {pkg} installed successfully!")
        break
    else:
        print(f"❌ {pkg}: {r.stderr.strip().split(chr(10))[-1]}")

# Try installing from GitHub source
print("\nTrying GitHub source install...")
r = subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/openmm/openmm-torch.git", "--quiet"],
    capture_output=True, text=True, timeout=300)
if r.returncode == 0:
    print("✅ openmm-torch installed from GitHub source!")
else:
    print(f"❌ GitHub source: {r.stderr.strip().split(chr(10))[-1]}")

# Verify
r2 = subprocess.run([sys.executable, "-c",
    "import openmmtorch; print(f'openmmtorch: OK')"],
    capture_output=True, text=True, timeout=10)
print(f"\nVerification: {r2.stdout.strip() if r2.stdout.strip() else r2.stderr.strip().split(chr(10))[-1]}")

✅ openmmtorch installed successfully!

Trying GitHub source install...
❌ GitHub source: ERROR: git+https://github.com/openmm/openmm-torch.git does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

Verification: openmmtorch: OK


In [15]:
# ============================================================================
# STEP 6 RETRY: Combined GPU Test Run (with openmmtorch now installed)
# ============================================================================
import subprocess, sys, os, datetime

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Step 6 — Combined GPU Test Run (retry): {timestamp_start}")
print("=" * 70)

env = os.environ.copy()
r = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_force_field_factory.py",
     "-k", "amoeba or ml_potential",
     "-v", "--tb=short"],
    capture_output=True, text=True, timeout=300,
    cwd="/content", env=env
)

# Show result lines
stdout_lines = r.stdout.strip().split("\n")
for line in stdout_lines:
    if any(kw in line for kw in ["PASSED", "FAILED", "SKIPPED", "===", "---"]):
        print(line)

print(f"\nReturn code: {r.returncode}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Step 6 completed: {timestamp_end}")

Step 6 — Combined GPU Test Run (retry): 2026-03-20 23:31
============================= test session starts ==============================
tests/test_force_field_factory.py::test_create_system_amoeba_requires_config PASSED [ 16%]
tests/test_force_field_factory.py::test_amoeba_system_produces_finite_energy PASSED [ 33%]
tests/test_force_field_factory.py::test_ml_potential_creates_valid_system PASSED [ 50%]
tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy PASSED [ 66%]
tests/test_force_field_factory.py::test_ml_potential_produces_finite_energy SKIPPED [ 83%]
tests/test_force_field_factory.py::test_ml_potential_import_error_message SKIPPED [100%]
=============================== warnings summary ===============================
============ 4 passed, 2 skipped, 4 deselected, 1 warning in 23.95s ============

Return code: 0
Step 6 completed: 2026-03-20 23:31


In [16]:
# ============================================================================
# STEP 7: Full Test Suite Regression on Colab
# Run the complete test suite to confirm 367+ passed, 0 failed
# Validates no previously passing test has regressed
# ============================================================================
import subprocess, sys, os, datetime

timestamp_start = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Step 7 — Full Test Suite Regression started: {timestamp_start}")
print("=" * 70)

env = os.environ.copy()
r = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/", "-v", "--tb=short"],
    capture_output=True, text=True, timeout=600,
    cwd="/content", env=env
)

# Print last 80 lines of stdout (full verbose output can be very long)
stdout_lines = r.stdout.strip().split("\n")
print(f"Total output lines: {len(stdout_lines)}")
print("\n--- Last 80 lines of pytest output ---")
for line in stdout_lines[-80:]:
    print(line)

if r.stderr:
    stderr_lines = r.stderr.strip().split("\n")
    important = [l for l in stderr_lines
                 if "WARN_EXTENSIONS" not in l and "UserWarning" not in l
                 and "warnings.warn" not in l and l.strip()]
    if important:
        print("\nSTDERR (filtered):")
        for l in important[-15:]:
            print(l)

print(f"\nReturn code: {r.returncode}")
timestamp_end = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Step 7 completed: {timestamp_end}")
print("=" * 70)

# Parse and highlight the summary line
for line in stdout_lines:
    if "passed" in line or "failed" in line or "error" in line:
        if "===" in line:
            print(f"\nFINAL SUMMARY: {line.strip()}")
            break

# Count results
passed = sum(1 for l in stdout_lines if " PASSED" in l)
failed = sum(1 for l in stdout_lines if " FAILED" in l)
skipped = sum(1 for l in stdout_lines if " SKIPPED" in l)
print(f"\nDetailed counts — Passed: {passed}, Failed: {failed}, Skipped: {skipped}")
if failed > 0:
    print("\n⚠️  REGRESSIONS DETECTED — investigate before proceeding")
    print("Failed tests:")
    for l in stdout_lines:
        if " FAILED" in l:
            print(f"  {l.strip()}")
else:
    print("\n✅ No regressions — all tests passed or were expectedly skipped")

Step 7 — Full Test Suite Regression started: 2026-03-20 23:31
Total output lines: 36

--- Last 80 lines of pytest output ---
============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.12.1, langsmith-0.7.18, typeguard-4.5.1
collecting ... collected 10 items

tests/test_force_field_factory.py::test_create_system_amber_produces_valid_system ERROR [ 10%]
tests/test_force_field_factory.py::test_create_system_rejects_unknown_family PASSED [ 20%]
tests/test_force_field_factory.py::test_create_system_amoeba_requires_config PASSED [ 30%]
tests/test_force_field_factory.py::test_create_system_ml_requires_config PASSED [ 40%]
tests/test_force_field_factory.py::test_amoeba_system_produces_finite_energy PASSED [ 50%]
tests/test_force_field_factory.py::test_ml_potential_creates_valid_system PASSED [ 60%]
tests/test_force_field_factory.py::t

In [17]:
# Step 7 Summary Extraction
stdout_lines = r.stdout.strip().split("\n")
passed = sum(1 for l in stdout_lines if " PASSED" in l)
failed = sum(1 for l in stdout_lines if " FAILED" in l)
skipped = sum(1 for l in stdout_lines if " SKIPPED" in l)
print(f"=== STEP 7 RESULTS ===")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Skipped: {skipped}")
print(f"Return code: {r.returncode}")
print()
# Show summary line
for line in stdout_lines:
    if "===" in line and ("passed" in line or "failed" in line):
        print(f"Summary: {line.strip()}")
# Show any FAILED tests
if failed > 0:
    print("\nFailed tests:")
    for l in stdout_lines:
        if " FAILED" in l:
            print(f"  {l.strip()}")
# Show last 10 lines
print("\n--- Last 10 lines ---")
for l in stdout_lines[-10:]:
    print(l)

=== STEP 7 RESULTS ===
Passed: 7
Failed: 0
Skipped: 2
Return code: 1

Summary: ============== 7 passed, 2 skipped, 1 warning, 1 error in 23.56s ===============

--- Last 10 lines ---
/content/tests/test_force_field_factory.py:87
=============================== warnings summary ===============================
tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy
  /usr/local/lib/python3.12/dist-packages/torchani/csrc/__init__.py:56: UserWarning: The extensions: ['cuaev', 'mnp', 'cell_list'] are not installed and will not be available. To install the extensions first install the CUDA Toolkit, and afterwards  run `ani build-extensions` To suppress warn set the env var TORCHANI_NO_WARN_EXTENSIONS=1 For example, if using bash, you may add `export TORCHANI_NO_WARN_EXTENSIONS=1` to your .bashrc
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=========================== short test summary info ============================
ERROR tes

In [18]:
# Check what files are on the Colab runtime
import subprocess, sys
r_ls = subprocess.run([sys.executable, "-c",
    "import os; "
    "print('\\n'.join(os.listdir('/content/tests/'))); "
    "print('---'); "
    "print('\\n'.join(os.listdir('/content/src/'))); "
    "print('---'); "
    "print('\\n'.join(os.listdir('/content/')))"],
    capture_output=True, text=True, timeout=10, cwd="/content")
print("=== /content/tests/ ===")
parts = r_ls.stdout.split("---")
print(parts[0].strip())
if len(parts) > 1:
    print("\n=== /content/src/ ===")
    print(parts[1].strip())
if len(parts) > 2:
    print("\n=== /content/ (top-level) ===")
    print(parts[2].strip())

=== /content/tests/ ===
__pycache__
conftest.py
test_force_field_factory.py
__init__.py

=== /content/src/ ===
__pycache__
physics
__init__.py
config.py

=== /content/ (top-level) ===
.config
tests
src
.pytest_cache
sample_data


In [19]:
# Investigate the ERROR on test_create_system_amber_produces_valid_system
stdout_lines = r.stdout.strip().split("\n")
for i, line in enumerate(stdout_lines):
    if "error" in line.lower() or "ERROR" in line:
        start = max(0, i-5)
        end = min(len(stdout_lines), i+10)
        for j in range(start, end):
            print(f"[{j}] {stdout_lines[j]}")
        print("---")

[2] cachedir: .pytest_cache
[3] rootdir: /content
[4] plugins: anyio-4.12.1, langsmith-0.7.18, typeguard-4.5.1
[5] collecting ... collected 10 items
[6] 
[7] tests/test_force_field_factory.py::test_create_system_amber_produces_valid_system ERROR [ 10%]
[8] tests/test_force_field_factory.py::test_create_system_rejects_unknown_family PASSED [ 20%]
[9] tests/test_force_field_factory.py::test_create_system_amoeba_requires_config PASSED [ 30%]
[10] tests/test_force_field_factory.py::test_create_system_ml_requires_config PASSED [ 40%]
[11] tests/test_force_field_factory.py::test_amoeba_system_produces_finite_energy PASSED [ 50%]
[12] tests/test_force_field_factory.py::test_ml_potential_creates_valid_system PASSED [ 60%]
[13] tests/test_force_field_factory.py::test_ml_potential_ani2x_direct_energy PASSED [ 70%]
[14] tests/test_force_field_factory.py::test_ml_potential_produces_finite_energy SKIPPED [ 80%]
[15] tests/test_force_field_factory.py::test_ml_potential_import_error_message SKIPPED [

In [20]:
# Concise error extraction - just the ERRORS section
stdout_lines = r.stdout.strip().split("\n")
in_errors = False
err_lines = []
for line in stdout_lines:
    if "ERRORS" in line and "===" in line:
        in_errors = True
    if in_errors:
        err_lines.append(line)
    if in_errors and ("short test summary" in line or ("===" in line and "passed" in line)):
        break

# Only print first 20 error lines
for l in err_lines[:20]:
    print(l)

==================================== ERRORS ====================================
_______ ERROR at setup of test_create_system_amber_produces_valid_system _______
file /content/tests/test_force_field_factory.py, line 87
  def test_create_system_amber_produces_valid_system(alanine_dipeptide_simulation):
E       fixture 'alanine_dipeptide_simulation' not found
>       available fixtures: anyio_backend, anyio_backend_name, anyio_backend_options, cache, capfd, capfdbinary, caplog, capsys, capsysbinary, capteesys, doctest_namespace, free_tcp_port, free_tcp_port_factory, free_udp_port, free_udp_port_factory, monkeypatch, pytestconfig, record_property, record_testsuite_property, record_xml_attribute, recwarn, tmp_path, tmp_path_factory, tmpdir, tmpdir_factory
>       use 'pytest --fixtures [testpath]' for help on them.

/content/tests/test_force_field_factory.py:87
=============================== warnings summary ===============================
tests/test_force_field_factory.py::test_ml_potent

In [21]:
# Ultra-compact error: just the error type and message
stdout_lines = r.stdout.strip().split("\n")
for line in stdout_lines:
    ll = line.strip()
    if ll.startswith("E ") or "Error" in ll and "===" not in ll:
        if len(ll) < 200:
            print(ll)

E       fixture 'alanine_dipeptide_simulation' not found
